# BioLGCA 1D


# Introduction

To simulate the dynamics for longer times and record the steps we use the class method `timeevo`. It takes the keyword arguments `timesteps`, that sets the number of time steps to simulate. Also, all configurations can be recorded using `record=True`. To only record the density profile set `recorddens=True`. The total number of cells can be recorded with `recordN=True`, which is also the default.

In [1]:
import matplotlib

import IPython
import ipykernel
import matplotlib_inline
import fipy
import sympy
print("SymPy version:", sympy.__version__)
print("FiPy version:", fipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("ipython:", IPython.__version__)
print("ipykernel:", ipykernel.__version__)
print("matplotlib_inline:", matplotlib_inline.__version__)

SymPy version: 1.14.0
FiPy version: 4.0.1
matplotlib: 3.10.6
ipython: 9.5.0
ipykernel: 6.30.1
matplotlib_inline: 0.1.7


In [2]:
%matplotlib notebook
from lgca import get_lgca

In [3]:
### use this in PyCharm
import matplotlib 
#matplotlib.use('Qt5Agg')
#import matplotlib_inline
%matplotlib inline
import matplotlib.pyplot as plt
#%matplotlib ipympl
#import matplotlib.pyplot as plt
import numpy as np

# Extra Code

In [ ]:
def update_oxygen_backwards_euler(O, cells, F, consumption_rate, dt):
    N = len(O)
    O_new = O.copy()


# build matrix A
# | 1+2F      -F  |           |... -2 1 0 0 ... |
# | -F       1+2F |    =      |... 1 -2 1 0 ... |
#                             |... 0 1 -2 1 ... | 
    main_diag = (1 + 2*F) * np.ones(N)
    off_diag  = -F * np.ones(N-1)

    # modify for BCs
    # first and  last row of matrix
    main_diag[0]  = 1 + F
    main_diag[-1] = 1 + F
    
    A = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

# what is b?? where does the consumption come into play?? does b = consumption? do i need to loop through N? 
#???????????????????????????
#    consumption = consumption_rate * cells * O * dt
#    b = O - consumption

    A += np.diag(consumption_rate * cells * dt)
    b = O.copy()
#???????????????????????????

    O_new = np.linalg.solve(A, b)

    return O_new

In [ ]:
def update_oxygen_backwards_euler_2(L, T, x, N):
    x = np.linspace(0, L, Nx+1)   # mesh points in space
    dx = x[1] - x[0]
    t = np.linspace(0, T, N+1)    # mesh points in time
    u   = np.zeros(Nx+1)          # unknown u at new time level
    u_n = np.zeros(Nx+1)          # u at the previous time level
    
    # Data structures for the linear system
    A = np.zeros((Nx+1, Nx+1))
    b = np.zeros(Nx+1)
    
    for i in range(1, Nx):
        A[i,i-1] = -F
        A[i,i+1] = -F
        A[i,i] = 1 + 2*F
    A[0,0] = A[Nx,Nx] = 1
    
    # Set initial condition u(x,0) = I(x)
    for i in range(0, Nx+1):
        u_n[i] = I(x[i])
    
    import scipy.linalg
    
    for n in range(0, Nt):
        # Compute b and solve linear system
        for i in range(1, Nx):
            b[i] = -u_n[i]
        b[0] = b[Nx] = 0
        u[:] = scipy.linalg.solve(A, b)
    
        # Update u_n before next step
        u_n[:] = u
        return u

In [ ]:
def update_oxygen_old(O, cells, F, consumption_rate, dt):
    N = len(O)
    O_new = O.copy()

# ghost points for boundaries
# index in O_ext:   0    1   2   3   ...   N   N+1
# meaning:        ghost O0  O1  O2  ...  O(N-1) ghost
    
    # ghost points
    O_ext = np.zeros(N + 2)
    O_ext[1:-1] = O
  #  O_ext[0] = O[1] # left ghost
  #  O_ext[-1] = O[-2] # right ghost
   
    O_ext[0] = O[2] # left ghost
    O_ext[-1] = O[-3] # right ghost

    # solve for SS, depends on consumption_rate and O[-1]
    # O_new = SS

    
    for i in range(N):
        #diffusion = F * (O[i-1] - 2*O[i] + O[i+1]) # without ghost points
        diffusion = F * (O_ext[i] - 2*O_ext[i+1] + O_ext[i+2]) # convert to ghost point index
        consumption = consumption_rate * cells[i] * O[i] * dt
       
        # michaelis menten:
    #    consumption = consumption_rate * cells[i] * O[i] / (Km + O[i]) * dt
        O_new[i] = O[i] + diffusion - consumption

    return O_new

In [ ]:
from fipy import Variable, FaceVariable, CellVariable, Grid1D, ExplicitDiffusionTerm, TransientTerm, DiffusionTerm, Viewer, LinearLUSolver
from fipy.tools import numerix
import numpy as np
import matplotlib.pyplot as plt

def oxygen_steady_state(lgca, O_init, cells, D, dx, consumption_rate, Km, dt, tmax):
    nx = len(lgca.oxygen)    
    mesh = Grid1D(nx=nx, dx=dx)
    O = CellVariable(name="Oxygen", mesh=mesh, value=lgca.oxygen.flatten())
    cells = CellVariable(name="cells", mesh=mesh, value=lgca.cell_density.flatten())
    
    # no flux
    O.faceGrad.constrain(0, mesh.facesLeft)
    # O.faceGrad.constrain(0, mesh.facesRight)
    # constant RHS oxygen source
    O.constrain(lgca.oxygen[-1], mesh.facesRight) 
    
    consumption = (O * cells * consumption_rate) / (Km + O)
    eq = TransientTerm() == DiffusionTerm(coeff=D) - consumption
   # eq.solve(var=O, dt=dt)
   # eq = DiffusionTerm(coeff=D) - consumption == 0
    eq.solve(var=O, dt=dt) 

   # timestep = dt
   # for i in range(tmax): # iterate until steady state
   #     old_O = O.value.copy()
  #      eq.solve(var=O, dt=timestep)
        #eq.solve(var=O, dt=timestep, solver=LinearLUSolver()) # implicit solver
  #      if np.max(np.abs(O.value - old_O)) < 1e-8: # if the difference is small, end loop
  #          break
    #print(O.value.copy()) #... steady state = IC right boundary
    #lgca.oxygen[:] = O.value
    return O.value.copy()

# f solve, and return output
# discretize using finite difference method, then 
#D / dx*2 * (c(n-1) + c(n-1) - 2(c(n+2))

In [ ]:
#def update_oxygen(O, cells, F, consumption_rate, dt):
#    O_new = O.copy()
 #   for i in range(1, len(O)-1): # interior points
#        O_new[i] = (O[i] + F*(O[i-1] - 2*O[i] + O[i+1]) - consumption_rate * cells[i] * O[i] * dt)
 #       O_new[0] = O_new[1] # no flux on left
 #   return O_new

# ghost points for boundaries
# index in O_ext:   0    1   2   3   ...   N   N+1
# meaning:        ghost O0  O1  O2  ...  O(N-1) ghost

#def update_oxygen(O, cells, F, consumption_rate, dt):
 #   N = len(O)
#    O_new = O.copy()
    
  #  O_ext = np.zeros(N + 2)
  #  O_ext[1:-1] = O
  #  O_ext[0] = O[1] # left ghost
  #  O_ext[-1] = O[-2] # right ghost

 #   Ix = range(1, len(O)-1)
  #  for i in range(N): # interior
 #   for i in Ix:
 #       diffusion = F * (O[i-1] - 2*O[i] + O[i+1]) 
       # diffusion = F * (O_ext[i] - 2*O_ext[i+1] + O_ext[i+2]) # convert to ghost point index       
 #       consumption = consumption_rate * cells[i] * O[i] * dt
 #       O_new[i] = O[i] + diffusion - consumption
    
    # reflecting boundaries
 #   O_new[0] = O_new[1] # left boundary
#    O_new[-1] = O_new[-2] # right boundary

    # dirichlet boundaries
   # O_new[0] = O[0]  # keep left source
  #  O_new[-1] = O[-1]  # keep right source

    # these boundaries skip over
   # i = Ix[0] # x=0 boundary
  #  O[i-1] = O[i+1]
   # i = Ix[-1] # x=L boundary
  #  O[i+1] = O[i-1]

 #   return O_new

In [ ]:
import numpy as np
import sys
#print(sys.executable)
from scipy.stats import truncnorm, truncexpon, expon
from copy import deepcopy
import numba
#print(numba.__version__)
from numba import jit
from lgca.interactions import tanh_switch

def go_or_grow_kappa_mod(lgca):
    """
    Apply the evolutionary "go-or-grow" interaction. Cells switch from a migratory to a resting phenotype and vice versa
    depending on their individual properties and the local cell density. Resting cells proliferate with a constant
    proliferation rate. Each cell dies with a constant rate. Daughter cells inherit their switch properties from the
    mother cells with some small variations given by a (truncated) Gaussian distribution.

    :param lgca: The lattice-gas cellular automata object.
    :return: None. The function modifies the lgca object in-place.
    """
    # Identify the relevant cells (those with non-zero density)
    relevant = (lgca.cell_density[lgca.nonborder] > 0)
    coords = [a[relevant] for a in lgca.nonborder]
    # Calculate the average density in the neighborhood
    nbdensity = lgca.nb_sum(lgca.cell_density, addCenter=True) / ((lgca.velocitychannels+1) * lgca.interaction_params['capacity']) # average density in neighborhood
    for coord in zip(*coords):
        node = lgca.nodes[coord]
        density = lgca.cell_density[coord]
        nbdens = nbdensity[coord]
        O_local = lgca.oxygen[coord]
        # rho = density / lgca.interaction_params['capacity']
        # Get the list of cells at the current node
        cells = np.array(node.sum())
        # R1: cell death
        # Determine which cells survive
        #notkilled = lgca.rng.random(size=density) < 1. - lgca.interaction_params['r_d']

        # cells die if in low oxygen for over 48 hrs
        if O_local < 1.419e-9: # micro-mol/cm
            print('mega low')
            lgca.anoxic_steps[cells] += 1
        else:
            lgca.anoxic_steps[cells] = 0
        notkilled = lgca.anoxic_steps[cells] < 1 # 34.8 hr (1.6 days) but reality is 48 hr...
        
        cells = cells[notkilled]
        # If all cells at the current node died, continue to the next node
        if len(cells) == 0:
            lgca.nodes[coord] = [[] for _ in range(lgca.K)]
            continue

        # Determine which cells switch phenotype based on their individual properties and the local cell density
        O_norm = O_local /  #lgca.oxygen.max()  # normalize to 0-1
        kappas = lgca.props['kappa'][cells]
        
       # switch_prob = tanh_switch(rho=nbdens, kappa=kappas, theta=lgca.interaction_params['theta'])
        switch_prob = tanh_switch(rho=O_norm, kappa=kappas, theta=lgca.interaction_params['theta'])

        switch = lgca.rng.random(len(cells)) < switch_prob
        restcells, velcells = list(cells[switch]), list(cells[~switch])
        # Update the density after deaths for birth
        rho = len(cells) / lgca.interaction_params['capacity']  # update density after deaths for birth
        # Determine the number of proliferating cells

        # if hypoxia, cells proliferate twice as slow
        if O_local < 1.419e-8:
            print('low')
            n_prolif = lgca.rng.binomial(len(restcells), max(0.5 * lgca.interaction_params['r_b'] * (1 - rho), 0))
        else:
            n_prolif = lgca.rng.binomial(len(restcells), max(lgca.interaction_params['r_b'] * (1 - rho), 0))
        
        
        #n_prolif = lgca.rng.binomial(len(restcells), max(lgca.interaction_params['r_b'] * (1 - rho), 0))
        # If there are proliferating cells, generate new cells
        if n_prolif > 0:
            proliferating = lgca.rng.choice(restcells, n_prolif, replace=False)
            lgca.maxlabel += n_prolif
            new_cells = np.arange(lgca.maxlabel - n_prolif + 1, lgca.maxlabel + 1)
            # Update the kappa properties of the new cells
            lgca.props['kappa'] = np.concatenate((lgca.props['kappa'],
                                                  lgca.rng.normal(loc=lgca.props['kappa'][proliferating],
                                                             scale=lgca.interaction_params['kappa_std'])))
            # Add the new cells to the list of resting cells
            restcells.extend(list(new_cells))

        # Initialize the node with empty channels and add the resting cells
        node = [[] for _ in range(lgca.velocitychannels)]
        node.append(restcells)
        # Assign the migrating cells to random velocity channels
        for cell in velcells:
            node[lgca.rng.integers(lgca.velocitychannels)].append(cell)

        # Update the node in the lgca object
        lgca.nodes[coord] = deepcopy(node)
      #  lgca.oxygen = update_oxygen(O, lgca.cell_density, F, consumption_rate, dt)
        lgca.oxygen = oxygen_ss_residual_sources(lgca, D, Km, consumption_rate, dx)
        lgca.update_dynamic_fields() #updates cell density


@jit(nopython=True)
def tanh_switch(rho, kappa=5., theta=0.8):
    return 0.5 * (1 + np.tanh(kappa * (rho - theta)))


@jit(nopython=True)
def nb_sum(qty, addCenter):
    sum = np.zeros(qty.shape)
    sum[:-1, ...] += qty[1:, ...]
    sum[1:, ...] += qty[:-1, ...]

    if addCenter:
        sum += qty
    return sum

In [ ]:
from scipy.optimize import root, least_squares

#ex:
#def residual(u):
#    uxx = (np.roll(u, -1) - 2*u + np.roll(u, 1)) / dx**2
#    uxx[0] = uxx[-1] = 0  # BCs
#    return D*uxx + r*u*(1-u)

#sol = root(residual, np.ones(N)*0.5)
#u_steady = sol.x

def oxygen_ss_residual(lgca, D, Km, consumption_rate, dx):

    cells = lgca.cell_density
    O_right = lgca.oxygen[-1]
    N = len(cells)
    dx = dx
    O_init = np.copy(lgca.oxygen)  # initial guess

    def residual(O):
        Oxx = np.zeros_like(O)
        # interior, O[0] is left boundary and 0[-1] is right boundary
        # discretize, U(j+1) - 2U(j) + U(j-1)
        Oxx[1:-1] = (O[2:] - 2*O[1:-1] + O[:-2]) / dx**2

        
        # Dirichlet, RHS
        O[-1] = O_right
        # Neumann no flux, LHS
        Oxx[0] = (2*O[1] - 2*O[0]) / dx**2

        R = np.zeros_like(O)
        R[:-1] = D * Oxx[:-1] - (O[:-1] * cells[:-1] * consumption_rate) / (Km + O[:-1])
        R[-1] = O[-1] - O_right
        return R
        
    # solve
    #sol = least_squares(residual, O_init, bounds=(0, np.inf)) # force lower bound so no negative oxygen
    sol = root(residual, O_init)
    return sol.x
    #if sol.success:
        #print(sol.x)
      #  return sol.x
  #  else:
      #  print("Root finding failed:", sol.message)
     #   return O_init


    #lgca update, update sources, update oxygen

In [ ]:
def update_oxygen_sources(lgca):

    # clog nodes
    deleted = lgca.mask & (lgca.cell_density > 0) 
    lgca.mask[deleted] = False

# Clog

Use def update_oxygen_sources_probability(lgca) (19.10.2026 AW)

In [4]:
def update_oxygen_sources_probability(lgca):

    # clog nodes
    #rho_node = lgca.cell_density / lgca.interaction_params['capacity']
    occupied = lgca.cell_density[1:-1] > 0
    clog_prob = 0.5 * (1.0 + np.tanh(lgca.interaction_params['kappa_clog'] * (lgca.cell_density[1:-1] - lgca.interaction_params['theta_clog'])))
    #clog_prob = tanh_switch(rho=rho_node, kappa=lgca.interaction_params['kappa_clog'], theta=lgca.interaction_params['theta_clog'])
    clogged = occupied & (lgca.rng.random(lgca.mask.shape) < clog_prob)
    #lgca.clog_mask[clogged] = True
    #occupied & (lgca.rng.random(rho_node.shape) < clog_prob)
    lgca.mask[clogged] = False

In [ ]:
def update_oxygen_sources_probability_0(lgca):
    prev_vessels = lgca.num_vessels_int.copy()

    occupied = lgca.cell_density > 0

    clog_prob = 0.5 * (1.0 + np.tanh(lgca.interaction_params['kappa_clog'] * (lgca.cell_density - lgca.interaction_params['theta_clog'])))

    # one clog per node
    clogged = (occupied & (lgca.rng.random(lgca.cell_density.shape) < clog_prob))

    # -1 vessel if clogged
    lgca.num_vessels_int[clogged] -= 1
    lgca.num_vessels_int[lgca.num_vessels_int < 0] = 0 # stay positive
    newly_zero = (prev_vessels > 0) & (lgca.num_vessels_int == 0)


    lgca.mask[clogged] = False

    return lgca.num_vessels_int, newly_zero

In [ ]:
def update_clogging_soft(lgca):
    occupied = lgca.cell_density > 0

    clog_prob = 0.5 * (1.0 + np.tanh(lgca.interaction_params['kappa_clog'] * (lgca.cell_density - lgca.interaction_params['theta_clog'])))

    clogged = occupied & (lgca.rng.random(lgca.cell_density.shape) < clog_prob)
    lgca.clog_mask[clogged] = True

    # instead of fully clogging, reduce vessel activity by 10% for clogged nodes
    lgca.vessel_activity[lgca.clog_mask] *= 0.9

# 1D

In [4]:
from scipy.optimize import root, least_squares
def oxygen_ss_residual_sources(lgca, D, Km, consumption_rate, dx):
    
    cells = lgca.cell_density[1:-1]
    O = lgca.oxygen
    #N = len(O)
    #O_right = lgca.oxygen_source
    #N = len(cells)
    O_init =  np.copy(lgca.oxygen)  # initial guess
    



    #update_oxygen_sources_probability(lgca)
    #if np.all(lgca.mask==False):
        #return np.zeros_like(lgca.oxygen)

    active_sources = lgca.mask #& (~lgca.clog_mask)
    inv_mask = ~active_sources
    O[active_sources] = lgca.oxygen_source
        

    def residual_sources(O):

        R = np.zeros_like(O)
        #R[active_sources] = O[active_sources] - lgca.oxygen_source
    

        #O[1:-1:4] = lgca.oxygen_source
        Oxx = np.zeros_like(O)
        # interior, O[0] is left boundary and 0[-1] is right boundary
        # discretize, U(j+1) - 2U(j) + U(j-1)... steady state uses second derivative
        Oxx[1:-1] = (O[2:] - 2*O[1:-1] + O[:-2]) / dx**2

        # Neumann no flux, LHS and RHS
        # Oxx[i]~O[i+1]-2O[i]+O[i-1]/dx^2
        # evaluate at i=0
        # ghost point: O[-1]=O[1] -> O[i+1]+O[i-1] becomes 2*O[1]
        # do the same on the left
        Oxx[0] = (2*O[1] - 2*O[0]) / dx**2
        Oxx[-1] = (2*O[-2] - 2*O[-1]) / dx**2
        
        
        #R[active_sources] = O[active_sources] - lgca.oxygen_source

        # only interior:
        R[inv_mask] = D * Oxx[inv_mask] - ((O[inv_mask] * cells[inv_mask] * consumption_rate) / (Km + O[inv_mask]))

        
        #R[:] = D * Oxx[:] - (O[:] * cells * consumption_rate) / (Km + O[:])
        #R[-1] = O[-1] - lgca.oxygen_source #1.316e-9*0.0025*0.0025*1e6*60
        #R[inv_mask] = D * Oxx[inv_mask] - ((O[inv_mask] * cells[inv_mask] * consumption_rate) / (Km + O[inv_mask]))
        #R[active_sources] = O[active_sources] - lgca.oxygen_source
        return R
        
    # solve
    #sol = least_squares(residual_sources, O_init, bounds=(0, np.inf)) # force lower bound so no negative oxygen
    sol = root(residual_sources, O_init,method='krylov')
    return sol.x


In [4]:
from scipy.optimize import root, least_squares
import numpy as np

def oxygen_ss_residual_sources_2(lgca, D, Km, consumption_rate, dx):

    cells = lgca.cell_density[1:-1]

    #clogged = cells > 5
    #lgca.mask[clogged] = False

    active_sources = lgca.mask
    inv_mask = ~active_sources

    # Initial full oxygen field
    O_init = np.copy(lgca.oxygen)

    # Enforce source values in the initial guess
    O_init[active_sources] = lgca.oxygen_source

    # Unknowns are ONLY the non-source nodes
    O0_free = O_init[inv_mask]

    def residual_free(O_free):

        # Reconstruct full oxygen field
        O = np.copy(O_init)

        # Fixed source nodes
        O[active_sources] = lgca.oxygen_source

        # Unknown nodes
        O[inv_mask] = O_free

        # Laplacian
        Oxx = np.zeros_like(O)

        Oxx[1:-1] = (
            O[2:] - 2 * O[1:-1] + O[:-2]
        ) / dx**2

        # Neumann boundaries
        Oxx[0] = (2 * O[1] - 2 * O[0]) / dx**2
        Oxx[-1] = (2 * O[-2] - 2 * O[-1]) / dx**2

        # Full residual
        R = np.zeros_like(O)

        R[inv_mask] = (
            D * Oxx[inv_mask]
            - (
                O[inv_mask]
                * cells[inv_mask]
                * consumption_rate
            ) / (Km + O[inv_mask])
        )

        # Return ONLY equations corresponding
        # to the unknowns
        return R[inv_mask]

    sol = root(residual_free, O0_free)
    #sol = least_squares(residual_free, O0_free, bounds=(0, np.inf))

    # Reconstruct full solution
    O_final = np.copy(O_init)
    O_final[active_sources] = lgca.oxygen_source
    O_final[inv_mask] = sol.x

    return O_final

# Non Steady State

In [ ]:
def update_ox(lgca, D, consumption_rate, dt, dx, Km):
    O = lgca.oxygen
    cells = lgca.cell_density
    N = len(O)

    O_new = np.copy(O)

    # left boundary
    lap_l = 2 * (O[1] - O[0]) / dx**2
    O_new[0] = O[0] + dt * (D * lap_l - (consumption_rate * cells[0] * O[0])/(Km + O[0]))

     # right boundary
    lap_r = 2 * (O[-2] - O[-1]) / dx**2
    O_new[-1] = O[-1] + dt * (D * lap_r - (consumption_rate * cells[-1] * O[-1])/(Km + O[-1]))

    for i in range(1, N-1):
        lap = (O[i-1] - 2*O[i] + O[i+1]) / dx**2
        O_new[i] = O[i] + dt * (D * lap - (consumption_rate * cells[i] * O[i])/(Km + O[i]))

    return O_new

In [ ]:
def update_ox_old(lgca, D, consumption_rate, dt, dx, Km):

    O = lgca.oxygen
    cells = lgca.cell_density
    N = len(O)

    O_ext = np.zeros(N + 2)
    O_ext[1:-1] = O
    O_ext[0] = O[1]
    O_ext[-1] = O[-2]

    lap = (O_ext[:-2] - 2*O_ext[1:-1] + O_ext[2:]) / dx**2

    reaction = (consumption_rate * cells * O) / (Km + O)

    O_new = O + dt * (D * lap - reaction)

    return O_new

In [ ]:
def update_ox_vessel(lgca, D, consumption_rate, dt, dx, Km):

    O = lgca.oxygen
    cells = lgca.cell_density
    N = len(O)

    mask = lgca.mask
    #inv = ~mask

    O_new = np.copy(O)

    update_oxygen_sources_probability(lgca)

    #O_new[mask] = lgca.oxygen_source
    active_sources = lgca.mask & (~lgca.clog_mask)
    O_new[active_sources] = lgca.oxygen_source

    # left boundary
    lap_l = 2 * (O[1] - O[0]) / dx**2
    O_new[0] = O[0] + dt * (D * lap_l - (consumption_rate * cells[0] * O[0])/(Km + O[0]))

     # right boundary
    lap_r = 2 * (O[-2] - O[-1]) / dx**2
    O_new[-1] = O[-1] + dt * (D * lap_r - (consumption_rate * cells[-1] * O[-1])/(Km + O[-1]))

    # only interior:
    for i in range(1, N-1):

        if active_sources[i]:
            continue  # skip source nodes completely

        lap = (O[i-1] - 2*O[i] + O[i+1]) / dx**2

        reaction = (consumption_rate * cells[i] * O[i]) / (Km + O[i])

        O_new[i] = O[i] + dt * (D * lap - reaction)

    return O_new

In [ ]:
def update_ox_vessel_cycle(lgca, D, consumption_rate, dt, dx, Km):

    O = lgca.oxygen
    cells = lgca.cell_density
    N = len(O)

    dt_oxygen = 2e-7
    nsteps = max(1, int(dt / dt_oxygen))
    dt_oxygen = dt / nsteps
    
    #update_oxygen_sources_probability(lgca)
    active_sources = lgca.mask & (~lgca.clog_mask)
    O[active_sources] = lgca.oxygen_source

    #O_new[mask] = lgca.oxygen_source


    for _ in range(nsteps):
        O_new = np.copy(O)


        # left boundary
        lap_l = 2 * (O[1] - O[0]) / dx**2
        O_new[0] = O[0] + dt_oxygen * (D * lap_l - (consumption_rate * cells[0] * O[0])/(Km + O[0]))

        # right boundary
        lap_r = 2 * (O[-2] - O[-1]) / dx**2
        O_new[-1] = O[-1] + dt_oxygen * (D * lap_r - (consumption_rate * cells[-1] * O[-1])/(Km + O[-1]))

        # only interior:
        for i in range(1, N-1):

            if active_sources[i]:
                continue  # skip source nodes completely

            lap = (O[i-1] - 2*O[i] + O[i+1]) / dx**2

            reaction = (consumption_rate * cells[i] * O[i]) / (Km + O[i])

            O_new[i] = O[i] + dt_oxygen * (D * lap - reaction)
    
        O = O_new
    lgca.oxygen = O

    return O

In [ ]:
def update_ox_vessel_cycle1(lgca, D, consumption_rate, dt, dx, Km):

    O = lgca.oxygen
    cells = lgca.cell_density

    active_sources = lgca.mask & (~lgca.clog_mask)
    diff_node = ~active_sources

    dt_oxygen = 2e-7
    nsteps = max(1, int(dt / dt_oxygen))
    dt_oxygen = dt / nsteps

    for _ in range(nsteps):

        # sources
        O[active_sources] = lgca.oxygen_source

        O_new = O.copy()

        Oxx = np.zeros_like(O)
        Oxx[1:-1] = (O[2:] - 2*O[1:-1] + O[:-2]) / dx**2

        # BC
        Oxx[0] = 2 * (O[1] - O[0]) / dx**2
        Oxx[-1] = 2 * (O[-2] - O[-1]) / dx**2

        reaction = (consumption_rate * cells * O) / (Km + O)

        # update only diff nodes aka places w diffusion 
        O_new[diff_node] = O[diff_node] + dt_oxygen * (D * Oxx[diff_node] - reaction[diff_node])

        #O_new[active_sources] = lgca.oxygen_source

        O = O_new

    lgca.oxygen = O
    return O

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np

def update_ox_vessel_ivp(lgca, D, consumption_rate, dt, dx, Km):

    cells = lgca.cell_density

    #update_oxygen_sources_probability(lgca)
    active_sources = lgca.mask & (~lgca.clog_mask)

    O0 = lgca.oxygen.copy()
    #O0[active_sources] = lgca.oxygen_source

    def rhs(t, O):

        O = O.copy()

        O[active_sources] = lgca.oxygen_source

        # Laplacian
        Oxx = np.zeros_like(O)
        Oxx[1:-1] = (O[2:] - 2*O[1:-1] + O[:-2]) / dx**2

        # Neumann boundaries
        Oxx[0] = 2 * (O[1] - O[0]) / dx**2
        Oxx[-1] = 2 * (O[-2] - O[-1]) / dx**2

        reaction = (consumption_rate * cells * O) / (Km + O)

        return D * Oxx - reaction

    #sol = solve_ivp(rhs,t_span=(0, dt),y0=O0,method="BDF", vectorized=False)
    sol = solve_ivp(rhs,t_span=(0, dt),y0=O0,method="BDF",rtol=1e-0, atol=1e-0)


    O_final = sol.y[:, -1]

    #O_final[active_sources] = lgca.oxygen_source

    lgca.oxygen = O_final
    return O_final

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np

def update_ox_vessel_ivp_jacobian(lgca, D, consumption_rate, dt, dx, Km):

    cells = lgca.cell_density

    #update_oxygen_sources_probability(lgca)
    active_sources = lgca.mask & (~lgca.clog_mask)

    O0 = lgca.oxygen.copy()
    N= len(O0)
    #O0[active_sources] = lgca.oxygen_source

    def rhs(t, O):

        O = O.copy()

        O[active_sources] = lgca.oxygen_source

        # Laplacian
        Oxx = np.zeros_like(O)
        Oxx[1:-1] = (O[2:] - 2*O[1:-1] + O[:-2]) / dx**2

        # Neumann boundaries
        Oxx[0] = 2 * (O[1] - O[0]) / dx**2
        Oxx[-1] = 2 * (O[-2] - O[-1]) / dx**2

        reaction = (consumption_rate * cells * O) / (Km + O)

        return D * Oxx - reaction

    def jac(t, O):

        N = len(O)
        J = np.zeros((N, N))

        r = consumption_rate * cells * Km / (Km + O)**2
        alpha = D / dx**2

        # interior diffusion
        for i in range(1, N-1):
            J[i, i-1] = alpha
            J[i, i]   = -2*alpha
            J[i, i+1] = alpha

        # boundaries (Neumann)
        J[0,0] = -2*alpha
        J[0,1] =  2*alpha

        J[-1,-1] = -2*alpha
        J[-1,-2] =  2*alpha

        # reaction (diagonal)
        J[np.arange(N), np.arange(N)] -= r

        # source nodes (Dirichlet constraint)
        J[active_sources, :] = 0.0
        J[active_sources, active_sources] = 1.0

        return J

    #sol = solve_ivp(rhs,t_span=(0, dt),y0=O0,method="BDF", vectorized=False)
    sol = solve_ivp(rhs,t_span=(0, dt),y0=O0,method="BDF",jac=jac)


    O_final = sol.y[:, -1]

    #O_final[active_sources] = lgca.oxygen_source

    lgca.oxygen = O_final
    return O_final

In [ ]:
def solver_BE(I, a, L, Nx, F, T, user_action=None):
    
    import time
    t0 = time.clock()

    x = linspace(0, L, Nx+1)   # mesh points in space
    dx = x[1] - x[0]
    dt = F*dx**2/a
    Nt = int(round(T/float(dt)))
    t = linspace(0, T, Nt+1)   # mesh points in time

    u   = zeros(Nx+1)   # solution array at t[n+1]
    u_1 = zeros(Nx+1)   # solution at t[n]

    # Representation of sparse matrix and right-hand side
    diagonal = zeros(Nx+1)
    lower    = zeros(Nx)
    upper    = zeros(Nx)
    b        = zeros(Nx+1)

    # Precompute sparse matrix
    diagonal[:] = 1 + 2*F
    lower[:] = -F  #1
    upper[:] = -F  #1
    # Insert boundary conditions
    diagonal[0] = 1
    upper[0] = 0
    diagonal[Nx] = 1
    lower[-1] = 0

    A = scipy.sparse.diags(
        diagonals=[diagonal, lower, upper],
        offsets=[0, -1, 1], shape=(Nx+1, Nx+1),
        format='csr')
    print A.todense()

    # Set initial condition
    for i in range(0,Nx+1):
        u_1[i] = I(x[i])

    if user_action is not None:
        user_action(u_1, x, t, 0)

    for n in range(0, Nt):
        b = u_1
        b[0] = b[-1] = 0.0  # boundary conditions
        u[:] = scipy.sparse.linalg.spsolve(A, b)

        if user_action is not None:
            user_action(u, x, t, n+1)

        # Update u_1 before next step
        #u_1[:] = u
        u_1, u = u, u_1

    t1 = time.clock()
    return u, x, t, t1-t0

In [ ]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve

def oxygen_backward_euler_mixed(lgca,D,dx,dt,Km,consumption_rate):
    """
    One backward-Euler oxygen diffusion step on a 1D lattice.

    Parameters
    ----------
    c : array, shape (N,)
        Current oxygen concentration.
    D : float
        Diffusion coefficient.
    dx : float
        Lattice spacing.
    dt : float
        Oxygen time step.
    source_mask : bool array, shape (N,)
        True where oxygen is fixed to source_value.
    source_value : float or array
        Fixed oxygen concentration at source sites.
    lambda_field : array, optional
        Consumption rate coefficient. Example: q * cell_occupancy.
    source_term : array, optional
        Additional production/source term on non-Dirichlet sites.
    boundary : str
        "noflux" or "dirichlet0".
    """
    #update_oxygen_sources_probability(lgca)
    cells = lgca.cell_density
    c = lgca.oxygen.copy()  
    N = len(c)
    Nt=50
    
    dt_sub = dt / Nt

    alpha = D * dt_sub / dx**2

    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    upper_diag = np.full(N - 1, -alpha)
    lower_diag = np.full(N - 1, -alpha)
    main_diag = np.full(N, 1 + 2 * alpha)

    # BC
    #main_diag[0] = 1 + alpha
    #main_diag[-1] = 1 + alpha
    upper_diag[0] = -2*alpha
    lower_diag[-1] = -2*alpha

    diagonals = [main_diag, upper_diag, lower_diag]
    offsets = [0, 1, -1]
    A_base = diags(diagonals, offsets, shape=(N, N), format='lil')
    
    source_index = np.where(lgca.mask)[0]
    for j in source_index:
        A_base[j, :] = 0.0
        A_base[j, j] = 1.0
        
    for step in range(Nt):
        reaction = - (c * consumption_rate * cells) / (Km + c)
        B = c + dt_sub * reaction
        B[source_index] = source_values[source_index]

        
        A_sparse = A_base.tocsr()

        c = spsolve(A_sparse, B)

    return c

# BE Extra

In [ ]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve

def oxygen_backward_euler(lgca,D,dx,dt,Km,consumption_rate, boundary="noflux"):
    """
    One backward-Euler oxygen diffusion step on a 1D lattice.

    Parameters
    ----------
    c : array, shape (N,)
        Current oxygen concentration.
    D : float
        Diffusion coefficient.
    dx : float
        Lattice spacing.
    dt : float
        Oxygen time step.
    source_mask : bool array, shape (N,)
        True where oxygen is fixed to source_value.
    source_value : float or array
        Fixed oxygen concentration at source sites.
    lambda_field : array, optional
        Consumption rate coefficient. Example: q * cell_occupancy.
    source_term : array, optional
        Additional production/source term on non-Dirichlet sites.
    boundary : str
        "noflux" or "dirichlet0".
    """
    cells = lgca.cell_density
    c = lgca.oxygen.copy()  
    lambda_field = consumption_rate * cells / (Km + c)
    #lambda_field = None
    #source_term = -(consumption_rate * cells *c)#/ (Km + c)
    source_term = None    
    N = len(c)

    #lgca.num_vessels_int, newly_zero = update_oxygen_sources_probability_0(lgca)

    if lambda_field is None:
        lambda_field = np.zeros(N)

    if source_term is None:
        source_term = np.zeros(N)

    alpha = D * dt / dx**2

    main = 1.0 + 2.0 * alpha + dt * lambda_field
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], offsets=[-1, 0, 1], shape=(N, N)).tolil()

    b = c + dt * source_term

    # Boundary conditions
    if boundary == "noflux":
        # Left boundary: c_{-1} = c_0
        # diffusion term becomes D(c_1 - c_0)/dx^2
        #A[0, 0] = 1.0 + alpha + dt * lambda_field[0]
        A[0, 1] = -2*alpha

        # Right boundary: c_N = c_{N-1}
        #A[N - 1, N - 1] = 1.0 + alpha + dt * lambda_field[N - 1]
        A[N - 1, N - 2] = -2*alpha


    # Internal oxygen sources: fixed concentration
    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    for j in np.where(lgca.mask)[0]:
        A[j, :] = 0.0
        A[j, j] = 1.0
        b[j] = source_values[j]
   # for j in np.where(newly_zero)[0]:
      #  A[j,:] = 0
      #  A[j,j] = 1
       # b[j] = source_values[j] * 0.7

    c_new = spsolve(A.tocsr(), b)

    return c_new

In [40]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve
from scipy.special import lambertw

def oxygen_backward_euler_op_split(lgca,D,dx,dt,Km,consumption_rate, boundary="noflux"):
    """
    One backward-Euler oxygen diffusion step on a 1D lattice.

    Parameters
    ----------
    c : array, shape (N,)
        Current oxygen concentration.
    D : float
        Diffusion coefficient.
    dx : float
        Lattice spacing.
    dt : float
        Oxygen time step.
    source_mask : bool array, shape (N,)
        True where oxygen is fixed to source_value.
    source_value : float or array
        Fixed oxygen concentration at source sites.
    lambda_field : array, optional
        Consumption rate coefficient. Example: q * cell_occupancy.
    source_term : array, optional
        Additional production/source term on non-Dirichlet sites.
    boundary : str
        "noflux" or "dirichlet0".
    """
    cells = lgca.cell_density
    c = lgca.oxygen.copy() 
    N = len(c)
    c_new = c.copy()

    #lgca.num_vessels_int, newly_zero = update_oxygen_sources_probability_0(lgca)

    alpha = D * dt / dx**2

    main = 1.0 + 2.0 * alpha
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], offsets=[-1, 0, 1], shape=(N, N)).tolil()

    b = c 

    # Boundary conditions
    if boundary == "noflux":
        # Left boundary: c_{-1} = c_0
        # diffusion term becomes D(c_1 - c_0)/dx^2
        #A[0, 0] = 1.0 + alpha + dt * lambda_field[0]
        A[0, 1] = -2*alpha

        # Right boundary: c_N = c_{N-1}
        #A[N - 1, N - 1] = 1.0 + alpha + dt * lambda_field[N - 1]
        A[N - 1, N - 2] = -2*alpha


    # Internal oxygen sources: fixed concentration
    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    for j in np.where(lgca.mask)[0]:
        A[j, :] = 0.0
        A[j, j] = 1.0
        b[j] = source_values[j]
    #for j in np.where(newly_zero)[0]:
       # A[j,:] = 0
       # A[j,j] = 1
      #  b[j] = source_values[j] * 0.7

    c_new = spsolve(A.tocsr(), b)
    #c_new_log = np.log(c_new)


    # explicit solve
    #o_ode = c_new * -(consumption_rate * cells) / (Km + c_new)

    exponent = (c_new - cells * consumption_rate * dt) / Km
    #print("max exponent:", np.max(exponent))
    #print("max c_new:", np.max(c_new))
    #print("min c_new:", np.min(c_new))
    #print("Km =", Km)
    #print("max uptake =", np.max(cells * consumption_rate * dt))
    #print("uptake/Km =", np.max(cells * consumption_rate * dt)/Km)
    
    diff_index = ~lgca.mask
    
    lambert_arg = (np.exp(exponent) * c_new/Km)
    lambert_function = Km * lambertw(lambert_arg)
    sol_real = lambert_function.real

    sol=sol_real.copy()
    sol[diff_index] = sol_real[diff_index]
    #sol[diff_index] = sol_real[diff_index]
    sol[lgca.mask] = lgca.oxygen_source # enforce sources again after reaction step



    return sol

In [ ]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve
from scipy.special import lambertw

def oxygen_backward_euler_op_split2(lgca,D,dx,dt,Km,consumption_rate, boundary="noflux"):

    c = lgca.oxygen.copy()
    cells = lgca.cell_density

    N = len(c)

    S = lgca.mask.astype(bool)   # source nodes
    F = ~S                       # free nodes

    source_values = (
        np.full(N, lgca.oxygen_source)
        if np.isscalar(lgca.oxygen_source)
        else np.asarray(lgca.oxygen_source)
    )

    alpha = D * dt / dx**2

    # ---------------------------
    # Build diffusion operator
    # ---------------------------
    main = (1.0 + 2.0 * alpha) * np.ones(N)
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], [-1, 0, 1], format="csr")

    # Neumann boundary corrections
    if boundary == "noflux":
        A = A.tolil()
        A[0, 1] = -2 * alpha
        A[-1, -2] = -2 * alpha
        A = A.tocsr()

    # ---------------------------
    # Split indices
    # ---------------------------
    F_idx = np.where(F)[0]
    S_idx = np.where(S)[0]

    # ---------------------------
    # RHS for diffusion
    # ---------------------------
    b = c.copy()

    # enforce Dirichlet coupling explicitly
    if len(S_idx) > 0:
        b[F_idx] = b[F_idx] - A[F_idx][:, S_idx].dot(source_values[S_idx])

    # ---------------------------
    # Reduced system (free nodes only)
    # ---------------------------
    if len(F_idx) > 0:
        A_FF = A[F_idx][:, F_idx]
        b_F = b[F_idx]
        c_F = spsolve(A_FF, b_F)
    else:
        c_F = np.array([])

    # reconstruct diffusion result
    c_new = c.copy()
    c_new[F_idx] = c_F
    c_new[S_idx] = source_values[S_idx]

    # ---------------------------
    # Reaction (only free nodes)
    # ---------------------------
    if consumption_rate != 0 and len(F_idx) > 0:

        cF = c_new[F_idx]
        cellsF = cells[F_idx]

        # Michaelis-Menten uptake via LambertW
        exponent = (cF - cellsF * consumption_rate * dt) / Km
        arg = np.exp(exponent) * cF / Km

        reacted = Km * lambertw(arg).real
        c_new[F_idx] = reacted

    # re-enforce Dirichlet (important)
    c_new[S_idx] = source_values[S_idx]

    return c_new

In [ ]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve
from scipy.integrate import solve_ivp

def oxygen_backward_euler_op_split3(lgca,D,dx,dt,Km,consumption_rate, boundary="noflux"):
    """
    One backward-Euler oxygen diffusion step on a 1D lattice.

    Parameters
    ----------
    c : array, shape (N,)
        Current oxygen concentration.
    D : float
        Diffusion coefficient.
    dx : float
        Lattice spacing.
    dt : float
        Oxygen time step.
    source_mask : bool array, shape (N,)
        True where oxygen is fixed to source_value.
    source_value : float or array
        Fixed oxygen concentration at source sites.
    lambda_field : array, optional
        Consumption rate coefficient. Example: q * cell_occupancy.
    source_term : array, optional
        Additional production/source term on non-Dirichlet sites.
    boundary : str
        "noflux" or "dirichlet0".
    """
    cells = lgca.cell_density
    c = lgca.oxygen.copy()   
    N = len(c)

    #lgca.num_vessels_int, newly_zero = update_oxygen_sources_probability_0(lgca)

    alpha = D * dt / dx**2

    main = 1.0 + 2.0 * alpha
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], offsets=[-1, 0, 1], shape=(N, N)).tolil()

    b = c 

    # Boundary conditions
    if boundary == "noflux":
        # Left boundary: c_{-1} = c_0
        # diffusion term becomes D(c_1 - c_0)/dx^2
        #A[0, 0] = 1.0 + alpha + dt * lambda_field[0]
        A[0, 1] = -2*alpha

        # Right boundary: c_N = c_{N-1}
        #A[N - 1, N - 1] = 1.0 + alpha + dt * lambda_field[N - 1]
        A[N - 1, N - 2] = -2*alpha


    # Internal oxygen sources: fixed concentration
    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    for j in np.where(lgca.mask)[0]:
        A[j, :] = 0.0
        A[j, j] = 1.0
        b[j] = source_values[j]

    c_star = spsolve(A.tocsr(), b)


    # explicit solve
    #def reaction_rhs(t, c, cells):
        #return -(consumption_rate * c * cells)/(Km + c)

    #sol = solve_ivp(reaction_rhs,(0, dt),c_star,args=(cells,),method='RK45')

    #c_new = sol.y[:, -1]

    reaction = -(consumption_rate * c_star * cells)/(Km + c_star)
    c_new = c_star + dt * reaction

    c_new[lgca.mask] = source_values[lgca.mask]

    return c_new

In [104]:
import numpy as np
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve

def oxygen_soft(lgca,D,dx,dt,Km,consumption_rate, boundary="noflux"):
    """
    One backward-Euler oxygen diffusion step on a 1D lattice.

    Parameters
    ----------
    c : array, shape (N,)
        Current oxygen concentration.
    D : float
        Diffusion coefficient.
    dx : float
        Lattice spacing.
    dt : float
        Oxygen time step.
    source_mask : bool array, shape (N,)
        True where oxygen is fixed to source_value.
    source_value : float or array
        Fixed oxygen concentration at source sites.
    lambda_field : array, optional
        Consumption rate coefficient. Example: q * cell_occupancy.
    source_term : array, optional
        Additional production/source term on non-Dirichlet sites.
    boundary : str
        "noflux" or "dirichlet0".
    """
    cells = lgca.cell_density
    c = lgca.oxygen.copy()  
    lambda_field = consumption_rate * cells / (Km + c)
    #lambda_field = None
    #source_term = -(consumption_rate * cells *c)#/ (Km + c)
    source_term = None    
    N = len(c)

    update_clogging_soft(lgca)
    if lambda_field is None:
        lambda_field = np.zeros(N)

    if source_term is None:
        source_term = np.zeros(N)

    alpha = D * dt / dx**2

    main = 1.0 + 2.0 * alpha + dt * lambda_field
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], offsets=[-1, 0, 1], shape=(N, N)).tolil()

    b = c + dt * source_term

    # Boundary conditions
    if boundary == "noflux":
        # Left boundary: c_{-1} = c_0
        # diffusion term becomes D(c_1 - c_0)/dx^2
        #A[0, 0] = 1.0 + alpha + dt * lambda_field[0]
        A[0, 1] = -2*alpha

        # Right boundary: c_N = c_{N-1}
        #A[N - 1, N - 1] = 1.0 + alpha + dt * lambda_field[N - 1]
        A[N - 1, N - 2] = -2*alpha


    # Internal oxygen sources: fixed concentration
    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    for j in range(N):
        a = lgca.vessel_activity[j]

        if a > 0.95:
            # strong Dirichlet
            A[j, :] = 0
            A[j, j] = 1
            b[j] = source_values[j]

        elif a > 0.1 and a <= 0.95:
            gamma = 10
            A[j, j] = 1.0
            b[j] = c[j] + dt * gamma * (source_values[j] - c[j])

    c_new = spsolve(A.tocsr(), b)

    return c_new

# BE

Use BE function: diffusion_only(lgca,D,dx,dt,Km,consumption_rate) (19.10.2026 AW)

In [ ]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

def diffusion_only(lgca,D,dx,dt,Km,consumption_rate):
    c=lgca.oxygen.copy()
    cells = lgca.cell_density[1:-1]
    N = len(c)
    alpha = D * dt / dx**2
    
    #update_oxygen_sources_probability(lgca)

    lambda_field = (consumption_rate * cells * Km) / ((Km + c)**2)
    #reaction term = consumption_rate * cells * c / (Km + c)

    #main = (1 + 2 * alpha) * np.ones(N)
    main = 1.0 + 2.0 * alpha + dt * lambda_field
    lower = -alpha * np.ones(N - 1)
    upper = -alpha * np.ones(N - 1)

    A = diags([lower, main, upper], offsets=[-1, 0, 1], format="lil")

    # Neumann boundary
    A[0, 1] = -2 * alpha
    A[-1, -2] = -2 * alpha

    A = A.tocsr()

    b = c.copy()
    if np.isscalar(lgca.oxygen_source):
        source_values = np.full(N, lgca.oxygen_source)
    else:
        source_values = np.asarray(lgca.oxygen_source)

    for j in np.where(lgca.mask)[0]:
        A[j, :] = 0.0
        #A.rows[j]=[j]
        A[j, j] = 1.0
        #A.data[j]=[1.0]
        b[j] = source_values[j]

    c_new = spsolve(A, b)
    return c_new

# Main

In [5]:
ufunclist = np.frompyfunc(list, 0, 1)
def get_arr_of_empty_lists(dims):
    return ufunclist(np.empty(dims, dtype=object))

In [6]:
from abc import ABC, abstractmethod
import matplotlib.colors as mcolors
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
import matplotlib.colors as colors
from matplotlib import cm
from numpy import random as npr
from sympy.utilities.iterables import multiset_permutations
from copy import copy, deepcopy
from lgca.plots import muller_plot
import warnings
from tqdm.auto import tqdm
def timeevo_mod(self, timesteps=200, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordnumvess=True, recordoxmask = True, recordchanneldens=True, showprogress=True, recordfampop=True):
        self.update_dynamic_fields()
        if record:
            self.nodes_t = get_arr_of_empty_lists((timesteps +1,) + self.dims + (self.K,))
            self.nodes_t[0, ...] = copy(self.nodes[self.nonborder])
        if recordN:
            self.n_t = np.zeros(timesteps + 1, dtype=np.uint)
            self.n_t[0] = self.cell_density[self.nonborder].sum()
        if recorddens:
            self.dens_t = np.zeros((timesteps + 1,) + self.dims, dtype=np.uint)
            self.dens_t[0, ...] = self.cell_density[self.nonborder]
        if recorddead:
            self.dead_dens_t = np.zeros((timesteps + 1,) + self.dims, dtype=np.uint)
            self.dead_dens_t[0, ...] = self.dead_cell_density[self.nonborder]
        if recordox:
            self.ox_t = np.zeros((timesteps + 1,) + self.dims)
            self.ox_t[0, ...] = self.oxygen #[self.nonborder]
        if recordnumvess:
            self.numvess_t = np.zeros((timesteps + 1,) + self.dims)
            self.numvess_t[0, ...] = self.num_vessels_int #[self.nonborder]
        if recordoxmask:
            self.oxmask_t = np.zeros((timesteps + 1,) + self.dims, dtype=bool)
            self.oxmask_t[0, ...] = self.mask #[self.nonborder]
        if recordchanneldens:
            self.channel_pop_t = np.zeros((timesteps + 1,) + self.dims + (self.K,), dtype=np.uint)
            self.channel_pop_t[0, ...] = self.channel_pop[self.nonborder]
        if recordfampop:
            from lgca.nove_ib_interactions import evo_steric
            # this needs to include all interactions that can increase the number of recorded families!
            if self.interaction in [evo_steric]:
                # if mutations are allowed, this is a list because it will be ragged due to increasing family numbers
                self.fam_pop_t = [self.calc_family_pop_alive()]
                is_mutating = True
            else:
                if 'family' not in self.props:
                    raise RuntimeError("Interaction does not deal with families, "
                                       "family population can therefore not be recorded.")
                # otherwise standard procedure
                self.fam_pop_t = np.zeros((timesteps + 1, self.maxfamily + 1))
                self.fam_pop_t[0, ...] = self.calc_family_pop_alive()
                is_mutating = False
                
        for t in tqdm(iterable=range(1, timesteps + 1), disable=1-showprogress):
            self.timestep()
 


            if record:
                self.nodes_t[t, ...] = copy(self.nodes[self.nonborder])
            if recordN:
                self.n_t[t] = self.cell_density[self.nonborder].sum()
            if recorddens:
                self.dens_t[t, ...] = self.cell_density[self.nonborder]
            if recorddead:
                self.dead_dens_t[t, ...] = self.dead_cell_density[self.nonborder]
            if recordox:
                self.ox_t[t, ...] = self.oxygen #[self.nonborder]
            if recordnumvess:
                self.numvess_t[t, ...] = self.num_vessels_int #[self.nonborder]
            if recordoxmask:
                self.oxmask_t[t, ...] = self.mask #[self.nonborder]
            if recordchanneldens:
                self.channel_pop_t[t, ...] = self.channel_pop[self.nonborder]
            if recordfampop:
                if is_mutating:
                    # append to the ragged nested list
                    self.fam_pop_t.append(self.calc_family_pop_alive())
                else:
                    # standard procedure
                    try:
                        self.fam_pop_t[t, ...] = self.calc_family_pop_alive()
                    except ValueError as e:
                        raise ValueError("Number of families has increased, interaction must be included in the case " +
                                         "distinction for the recordfampop keyword in the IBLGCA base timeevo function!") from e
        if recordfampop and is_mutating:
            self.straighten_family_populations()

In [7]:
def softmax(x):
    x = np.asarray(x)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

In [ ]:
import numpy as np
import sys
#print(sys.executable)
from scipy.stats import truncnorm, truncexpon, expon
from copy import deepcopy
import numba
#print(numba.__version__)
from numba import jit
from lgca.interactions import tanh_switch

def go_or_grow_kappa_chemo_mod(lgca):
    """
    Apply the evolutionary "go-or-grow" interaction. Cells switch from a migratory to a resting phenotype and vice versa
    depending on their individual properties and the local cell density. Resting cells proliferate with a constant
    proliferation rate. Migrating cells move along the cell density gradient.
    Each cell dies with a constant rate. Daughter cells inherit their switch properties from the
    mother cells with some small variations given by a (truncated) Gaussian distribution.
    :param lgca:
    :return:
    """
    relevant = (lgca.cell_density[lgca.nonborder] > 0) # g/.et sites that have cells, nonborder means we only want cells inside (and not on the border)
    coords = [a[relevant] for a in lgca.nonborder] # get the coordinates of the sites that have cells
  #  g = lgca.gradient(lgca.cell_density / lgca.interaction_params['capacity'])  # density gradient for each lattice site, normalized by the capacity (max number of cells per site)
    
    g = lgca.gradient(lgca.oxygen / lgca.interaction_params['O_norm'])  # oxygen gradient for each lattice site, normalized by the total oxygen
    # change ox max to be something constant, find biologically high value of O2 in blood
    
    #  nbdensity = lgca.nb_sum(lgca.cell_density, addCenter=True) / (lgca.velocitychannels * lgca.interaction_params['capacity'])
    #nbdensity = lgca.nb_sum(lgca.cell_density + lgca.cell_density) / (lgca.velocitychannels * lgca.interaction_params['capacity']) # density of neighbors for each site

    O2low = lgca.oxygen < lgca.interaction_params['lowox'] #2.303e-7
    O2ultralow = lgca.oxygen < lgca.interaction_params['ultralowox'] #2.303e-8

    
    for coord in zip(*coords): # for each coordinate that is "relevant" (has cells in border)
        node = lgca.nodes[coord] # list of channels for each site, each containing cell IDs
        density = lgca.cell_density[coord] # number of cells at the site
        #nbdens = nbdensity[coord] # average density of neighbors
        O_local = lgca.oxygen[coord]
        rho = density / lgca.interaction_params['capacity'] # density of the site relative to the capacity
        cells = np.array(node.sum()) # get a 1D array of cell IDs at this site
        # R1: cell death
        #notkilled = lgca.rng.random(size=density) < 1. - lgca.interaction_params['r_d'] # mask; true if cell lives and false if cell dies

        # cells die if in low oxygen for over 48 hrs
        if O2ultralow[coord]:
            notkilled = lgca.rng.random(size=density) < 1. - lgca.interaction_params['r_d_hypoxia']
        else:
            notkilled = lgca.rng.random(size=density) < 1. - lgca.interaction_params['r_d']

        # track killed cells
        killed = ~notkilled
        #num_killed = np.count_nonzero(killed)
        #num_alive = np.count_nonzero(notkilled)
        #assert num_alive + num_killed == density
        #node_num = int(coord[0])
        #print(f"Coord {node_num}: total={density}, killed={num_killed}, alive={num_alive}, cumulative_dead={lgca.dead_cell_density[coord] + num_killed}")
        lgca.dead_cell_density[coord] += killed.sum()
        #print(f" -> Updated cumulative dead at {node_num}: {lgca.dead_cell_density[coord]}")


        cells = cells[notkilled] # update cells to only have the alive cells
        if len(cells) == 0: # if all cells are dead
            lgca.nodes[coord] = [[] for _ in range(lgca.K)] # reset the node to an empty list for each velocity channel
            continue # and then skip the rest of this loop

        # Determine which cells switch phenotype based on their individual properties and the local cell density
        #O_norm = O_local / lgca.interaction_params['max_ox']  # normalize to 0-1
        O_norm = O_local / lgca.interaction_params['O_norm']  # normalize to 0-1, 24mmhg

        kappas = lgca.props['kappa'][cells]
        thetas = lgca.props['theta'][cells]
        
      #  switch = lgca.rng.random(len(cells)) < tanh_switch(rho=nbdens, kappa=kappas, theta=lgca.interaction_params['theta']) # switch if less than probability tanh_switch; tanh_switch depends on neighbor denisty
        switch = lgca.rng.random(len(cells)) < tanh_switch(rho=O_norm, kappa=kappas, theta=thetas) # switch if less than probability tanh_switch; tanh_switch depends on oxygen
        # print(len(cells) == len(switch))
        restcells, velcells = list(cells[switch]), list(cells[~switch]) # restcells are those that switch and grow, velcells are those that don't switch and migrate

        rho = len(cells) / lgca.interaction_params['capacity']  # update density after deaths for birth

        # if hypoxia, cells proliferate twice as slow
        if O2low[coord]: #O_local < 1.419e-8:
            n_prolif = lgca.rng.binomial(len(restcells), max(0.5 * lgca.interaction_params['r_b'] * (1 - rho), 0))
        else:
            n_prolif = lgca.rng.binomial(len(restcells), max(lgca.interaction_params['r_b'] * (1 - rho), 0))

        
        if n_prolif > 0: # if there are cells to proliferate
            proliferating = lgca.rng.choice(restcells, n_prolif, replace=False) # randomly choose n_prolif cells to divide
            lgca.maxlabel += n_prolif # update the maximum by adding the number of n_prolif
            new_cells = np.arange(lgca.maxlabel - n_prolif + 1, lgca.maxlabel + 1) # assign new labels to these cells; call them new_cells
            lgca.props['kappa'] = np.concatenate((lgca.props['kappa'],
                                                  lgca.rng.normal(loc=lgca.props['kappa'][proliferating],
                                                             scale=lgca.interaction_params['kappa_std']))) # daughter cells inherit kappa from parent cell +/- a kappa standard deviation
            lgca.props['theta'] = np.concatenate((lgca.props['theta'],
                                                  np.clip(lgca.rng.normal(loc=lgca.props['theta'][proliferating],
                                                            scale=lgca.interaction_params['theta_std']),0,1)))
            
            restcells.extend(list(new_cells)) # add the new_cells to restcells group

        node = [[] for _ in range(lgca.velocitychannels)] # create a new node structure; this starts empty
        node.append(restcells) # this adds restcells to the new node
        if len(velcells) > 0: # if there are cells that are migrating
            gloc = g[coord] # get the gradient at this coordinate
            scores = (lgca.interaction_params['beta'] * np.einsum('i,ij', gloc, lgca.c)) # weights = migration probabilities in each velocity direction

            weights = softmax(scores)
            weights = weights.ravel()
            
            z = weights.sum() # sum all weights (migration probabilities)
            weights /= z  # normalize so that these weights = 1
          
            sample = lgca.rng.multinomial(len(velcells), weights) # array of integers from a multinomial distribution, telling how many migratory cells go into each velocity channel
            lgca.rng.shuffle(velcells) # shuffle the migrating cells
            for i in range(lgca.velocitychannels): # loop over velocity channels (directions in which cells can move)
                node[i].extend(velcells[:sample[i]]) # node[i] = cells currently assigned to velocity channel i at this lattice site; take sample[i] and add these cells to the ith velocity channel
                velcells = velcells[sample[i]:] # velcells contains only the remaining cells to assign to the next channels

        lgca.nodes[coord] = node # save the updated node back to the lattice
        
    lgca.update_dynamic_fields()
   
    # these are right 7/12
    lgca.oxygen = diffusion_only(lgca,D,dx,dt,Km,consumption_rate)


@jit(nopython=True)
def tanh_switch(rho, kappa=5., theta=0.8):
    return 0.5 * (1 + np.tanh(kappa * (rho - theta)))


@jit(nopython=True)
def nb_sum(qty, addCenter):
    sum = np.zeros(qty.shape)
    sum[:-1, ...] += qty[1:, ...]
    sum[1:, ...] += qty[:-1, ...]

    if addCenter:
        sum += qty
    return sum

In [9]:
l = 2000
tmax = 2000
restchannels = 1
n_channels = restchannels + 2

# set PDE parameters
width = 0.0025
D = 13.824 # cm2/day # diffusion coefficient
dx = width # lattice step in space, 25 µm
dt = 1.6 # changed 1.6 # time step, 1.6 days

# convert!!
two_d_vol=width**2
convert_const = 1.316e-9*two_d_vol*1e6

consumption_rate = 8.64e-7/width # µmol cell-1 day-1 cm-1
print(consumption_rate)

# if Michaelis-Menten
Km = convert_const*2.5 # micro-mol/cm

nodes = np.zeros((l, n_channels))
capacity_param = 12 #(100/8)
k=8
nodes[int((l/2)-(k/2)): int((l/2)+(k/2))] = capacity_param/3 # capacity, start cells at left

0.0003456


In [10]:
#print(tanh_switch(rho=0.6484295845997974, kappa=1.106, theta=0.4498))
print(convert_const*60)
print((convert_const*0.7)/(convert_const*60))

4.935e-07
0.011666666666666667


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import types
import os    
import pandas as pd

output_dir = "08212026_no_clog_runs25to50"
os.makedirs(output_dir, exist_ok=True)
#lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
#lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=True, recordfampop=False)
#O_history = lgca.ox_t


#O_final = lgca.ox_t[-1]   # last timestep, shape: (Nr_new,)
#print(lgca.oxygen)
#nodes_to_check = [1, 10, 50, 100, 150, 200, 250, 300, 400, 499]
    
#print(f"\nRun final oxygen values:")
#for n in nodes_to_check:
    #if n < len(O_final):
        #print(f"Node {n}: {O_final[n]}")
    #else:
        #print(f"Node {n}: out of range (max {len(O_final)-1})")

run_id=25
n_repeats=10

for rep in range(n_repeats):
    run_id += 1
    print(run_id)


    kappa_min = -2
    kappa_max = 2
    theta_min = 0
    theta_max = 1
    kappa_scan = np.random.uniform(kappa_min, kappa_max)
    theta_scan = np.random.uniform(theta_min, theta_max)
    lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
    
    # these give default values
    lgca.interaction_params['capacity'] = capacity_param
    lgca.interaction_params['kappa_std'] = 0.2
    lgca.interaction_params['r_d'] = 0.01 #* 0.25 # changed
    lgca.interaction_params['r_b'] = 1 #* 0.25 #0.2 #changed

    lgca.interaction_params['beta'] = 0.5
    #lgca.oxygen = np.zeros(lgca.cell_density.shape)
    lgca.oxygen = np.zeros(lgca.dims)
    lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)
    
    # grid mask
    #lgca.mask = np.zeros_like(lgca.oxygen, dtype=bool)
    #lgca.mask[1:-1:4] = True
    
    # random mask
    nx_full = lgca.cell_density.shape[0]
    #print(nx_full)
    lgca.mask = np.zeros((nx_full-2), dtype=bool)
    p_vessel = 0.06
    #p_vessel = 0.01
    #lgca.mask[1:-1]= False
    #lgca.mask[1000] = True
    lgca.mask = lgca.rng.random((nx_full-2)) < p_vessel
    lgca.original_mask = lgca.mask.copy()
    lgca.clog_mask = np.zeros_like(lgca.mask, dtype=bool)
    lgca.num_vessels_int = np.zeros(nx_full-2, dtype=int)
    lgca.num_vessels_int[lgca.mask] = 1
    lgca.vessel_activity = lgca.mask.astype(float)

    
    lgca.oxygen[:] = convert_const*60
    #convert_const*24 # µmol/cm
    #lgca.oxygen[-1]=convert_const*60
    lgca.oxygen_source = convert_const* 60 #60 # µmol/cm

    lgca.interaction_params['O_norm'] = convert_const*60
    
    lgca.interaction_params['r_d_hypoxia'] = 0.5 #* 0.25 # changed 
    lgca.interaction_params['max_ox'] = convert_const*95 # µmol/cm
    
    lgca.interaction_params['lowox'] = convert_const*7
    lgca.interaction_params['ultralowox'] = convert_const*0.7
    
    lgca.interaction_params['kappa_clog'] = 0.5 #0.3
    lgca.interaction_params['theta_clog'] = 3 #12
    
    # choose interaction
    lgca.interaction = go_or_grow_kappa_chemo_mod

    lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
    lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordnumvess=True, recordoxmask=True, recordchanneldens=True, showprogress=False, recordfampop=False)
    O_history = lgca.ox_t
    
    title_str = f"{run_id}"
    filename_tag = title_str
    
                    
    plt.figure()
    plt.imshow(O_history, aspect='auto', origin='upper', cmap='magma')
    plt.colorbar(label='Oxygen')
    plt.xlabel('Lattice node')
    plt.ylabel('Time step')
    plt.title(fr'Oxygen {title_str}')
    plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
    plt.close()
                    
    plt.figure()
    lgca.plot_density(cmap='cividis')
    plt.title(fr'Cell Density {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
    plt.close()
            
    plt.figure()
    lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='bwr',vmin=-4,vmax=4)
    cs = plt.contour(O_history,levels=1,colors='black',linewidths=1)
    plt.title(fr'κ Distribution {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
    plt.close()
            
    plt.figure()
    lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='bwr',vmin=0,vmax=1)
    cs = plt.contour(O_history,levels=1,colors='black',linewidths=1)
    plt.title(fr'θ Distribution {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
    plt.close()
            
    plt.figure()
    lgca.plot_density(lgca.dead_dens_t,cmap='cividis')
    plt.title(fr'Dead Cell Density {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
    plt.close()
                    
    plt.figure()
    lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='cividis')
    plt.title(fr'Proliferating Cell Density {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
    plt.close()
            
    plt.figure()
    lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='cividis')
    plt.title(fr'Migrating Cell Density {title_str}')
    plt.xlim(0, l)
    plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
    plt.close()


    migrating = lgca.channel_pop_t[..., :-1].sum(axis=-1)
    plt.figure()
    # migration heatmap (matrix coordinates: (0,0) top-left)
    lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='cividis')
    # oxygen contours (same grid)
    cs = plt.contour(O_history,levels=1,colors='white',linewidths=1)
    plt.title(fr'Migrating Cell Density + Oxygen Contours {title_str}')
    plt.xlim(0, l)
    plt.savefig(
    f"{output_dir}/migration_oxygen_overlay_{filename_tag}.png",dpi=300)
    plt.close()


    proliferating = lgca.channel_pop_t[..., :-1].sum(axis=-1)
    plt.figure()
    # migration heatmap (matrix coordinates: (0,0) top-left)
    lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='cividis')
    # oxygen contours (same grid)
    cs = plt.contour(O_history,levels=1,colors='white',linewidths=1)
    plt.title(fr'Proliferating Cell Density + Oxygen Contours {title_str}')
    plt.xlim(0, l)
    plt.savefig(
    f"{output_dir}/proliferating_oxygen_overlay_{filename_tag}.png",dpi=300)
    plt.close()
    
    # kappa histogram
    #mean_kappa = lgca.mean_prop_t['kappa']
    #all_vals_k = []
    #for t in range(tmax-50, tmax):
        #vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
        ##print(f"t={t}, vals_k size={len(vals_k)}")
        #all_vals_k.append(vals_k)
    #all_vals_k = np.concatenate(all_vals_k)

    mean_kappa = lgca.mean_prop_t['kappa']
    density_count = lgca.dens_t
    all_vals_k = []
    for t in range(tmax-50, tmax):
        k = mean_kappa[t]
        d = density_count[t]
        # handle masked arrays
        if hasattr(k, "compressed"):
            k = k.filled(np.nan)
        if hasattr(d, "compressed"):
            d = d.filled(0)
        mask = d > 0
        vals_k = k[mask]
        all_vals_k.append(vals_k)
    all_vals_k = np.concatenate(all_vals_k)

    
    plt.figure()
    plt.hist(all_vals_k, bins=30, linewidth=1)
    plt.xlabel("κ")
    plt.ylabel("Count")
    plt.title((fr'Kappa Distribution {title_str}'))
    plt.tight_layout()
    plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
    plt.close()
                    
                    
    # theta histogram
    #mean_theta = lgca.mean_prop_t['theta']
    #all_vals_t = []
    #for t in range(tmax-50, tmax):
        #vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
        #all_vals_t.append(vals_t)
    #all_vals_t = np.concatenate(all_vals_t)

    mean_theta = lgca.mean_prop_t['theta']
    all_vals_t = []
    for t in range(tmax-50, tmax):
        th = mean_theta[t]
        d = density_count[t]
        if hasattr(th, "compressed"):
            th = th.filled(np.nan)
        if hasattr(d, "compressed"):
            d = d.filled(0)
        mask = d > 0
        vals_t = th[mask]
        all_vals_t.append(vals_t)
    all_vals_t = np.concatenate(all_vals_t)
    
    plt.figure()
    plt.hist(all_vals_t, bins=30, linewidth=1)
    plt.xlabel("θ")
    plt.ylabel("Count")
    plt.title((fr'Theta Distribution {title_str}'))
    plt.tight_layout()
    plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
    plt.close()
    
    # heat map
    # range on plot
    #min_kappa = min(all_vals_k.min(), kappa_scan)
    #max_kappa = max(all_vals_k.max(), kappa_scan)
    #min_theta = min(all_vals_t.min(), theta_scan)
    #max_theta = max(all_vals_t.max(), theta_scan)


    # overall heatmap
    plt.figure()
    counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[-4, 4], [0, 1]], cmap='viridis')
    plt.xlabel("κ")
    plt.ylabel("θ")
    #plt.xlim(min_kappa, max_kappa)
    #plt.ylim(min_theta, max_theta)
    plt.title(f"κ–θ Heat Map {title_str}")
    plt.colorbar(im, label="Frequency")
    #plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
    plt.close()

    # separate heat maps
    oxygen_vals = []
    for t in range(tmax-50, tmax):
        ox = lgca.ox_t[t] / lgca.interaction_params['O_norm']
        d = density_count[t]
        if hasattr(ox, "compressed"):
            ox = ox.filled(np.nan)
        if hasattr(d, "compressed"):
            d = d.filled(0)
        # only occupied nodes
        mask_1 = d > 0
        oxygen_vals.append(ox[mask_1])
    oxygen_vals = np.concatenate(oxygen_vals)
    # split by normalized oxygen
    low_mask = oxygen_vals < 0.117
    high_mask = oxygen_vals >= 0.117

    # low heatmap
    plt.figure(figsize=(6,5))
    counts, xedges, yedges, im = plt.hist2d( all_vals_k[low_mask],all_vals_t[low_mask],bins=80,range=[[-4,4],[0,1]],cmap='viridis')
    plt.xlabel("κ")
    plt.ylabel("θ")
    plt.title(rf'$\kappa$–$\theta$ Heatmap for $\rho < 0.117$ {title_str}')
    plt.colorbar(im, label='Frequency')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/kappa_theta_heatmap_lowrho_{filename_tag}.png",dpi=300)
    plt.close()

    # high heatmap
    plt.figure(figsize=(6,5))
    counts, xedges, yedges, im = plt.hist2d( all_vals_k[high_mask],all_vals_t[high_mask],bins=80,range=[[-4,4],[0,1]],cmap='viridis')
    plt.xlabel("κ")
    plt.ylabel("θ")
    plt.title(rf'$\kappa$–$\theta$ Heatmap for $\rho ≥ 0.117$ {title_str}')
    plt.colorbar(im, label='Frequency')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/kappa_theta_heatmap_highrho_{filename_tag}.png",dpi=300)
    plt.close()


    
    
    # last time step, proliferating vs migrating 
    last_migrating = lgca.channel_pop_t[-1, :, :-1].sum(axis=-1) 
    last_proliferating = lgca.channel_pop_t[-1, :, -1] 
    total_mp = last_migrating + last_proliferating 
    pct_migrating_last = np.zeros_like(total_mp, dtype=float)
    pct_proliferating_last = np.zeros_like(total_mp, dtype=float)
    np.divide(last_migrating, total_mp, out=pct_migrating_last, where=total_mp != 0)
    np.divide(last_proliferating, total_mp, out=pct_proliferating_last, where=total_mp != 0)
    pct_migrating_last *= 100
    pct_proliferating_last *= 100
    nodes_mp = np.arange(len(pct_migrating_last)) 
    plt.figure() 
    plt.plot(nodes_mp, pct_migrating_last, label='Migrating (%)') 
    plt.plot(nodes_mp, pct_proliferating_last, label='Proliferating (%)') 
    plt.xlabel('Lattice node') 
    plt.ylabel('Percentage of cells') 
    plt.legend() 
    plt.tight_layout() 
    plt.savefig(f"{output_dir}/migration_vs_proliferation_end_{filename_tag}.png", dpi=300) 
    plt.close()
    
    # last time step, kappa
    kappa_spatial = lgca.calc_prop_mean(nodes=lgca.nodes_t[-1],props=lgca.props,propname='kappa')
    nodes_ka = np.arange(kappa_spatial.shape[0])
    plt.figure()
    plt.plot(nodes_ka, kappa_spatial)
    plt.xlabel('Lattice node')
    plt.ylabel(r'$\kappa$')
    plt.title(fr'κ Spatial Profile {title_str}')
    plt.xlim(0, l)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/kappa_end_{filename_tag}.png", dpi=300) 
    plt.close()


    
    last_few = tmax-50
    cell_density_txt = lgca.dens_t[last_few:tmax]
    oxygen_txt = lgca.ox_t[last_few:tmax]
    kappa_txt = np.array([lgca.calc_prop_mean(nodes=lgca.nodes_t[t],props=lgca.props,propname='kappa') for t in range(last_few, tmax)])
    theta_txt = np.array([lgca.calc_prop_mean(nodes=lgca.nodes_t[t],props=lgca.props,propname='theta') for t in range(last_few, tmax)])
    last_migrating_txt = lgca.channel_pop_t[last_few:tmax, :, :-1].sum(axis=-1) 
    last_proliferating_txt = lgca.channel_pop_t[last_few:tmax, :, -1] 
    nodes_txt = np.arange(cell_density_txt.shape[1])
    time_txt = np.arange(last_few, tmax)
    
    T, X = np.meshgrid(time_txt, nodes_txt, indexing='ij')
    
    df = pd.DataFrame({
       "time": T.ravel(),
        "node": X.ravel(),
        "density": cell_density_txt.ravel(),
        "oxygen": oxygen_txt.ravel(),
        "kappa": kappa_txt.ravel(),
        "theta": theta_txt.ravel(),
        "migrating": last_migrating_txt.ravel(),
        "proliferating": last_proliferating_txt.ravel(),})
    
    df = df.sort_values(by=["time", "node"], ascending=[False, True])
   
    output_file = f"{output_dir}/lgca_1d_no_clogging.xlsx"
    sheet_name = f"run_{run_id}"
    
    with pd.ExcelWriter(output_file, engine="openpyxl", mode="a" if os.path.exists(output_file) else "w") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)


26


/opt/miniconda3/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


27


/opt/miniconda3/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


28


...

/opt/miniconda3/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)
/opt/miniconda3/lib/python3.13/site-packages/matplotlib/colors.py:778: RuntimeWarning: overflow encountered in multiply
  xa *= self.N


30


/opt/miniconda3/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


31


/opt/miniconda3/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


# More Extra Code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import types
import os
lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=True, recordfampop=False)
O_history = lgca.ox_t

In [ ]:
# uniform dist of kappas and thetas
kappa_min = -2
kappa_max = 2
theta_min = 0
theta_max = 1
kappa_scan = np.random.uniform(kappa_min, kappa_max)
theta_scan = np.random.uniform(theta_min, theta_max)
lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)

# these give default values
lgca.interaction_params['capacity'] = capacity_param
lgca.interaction_params['kappa_std'] = 0.2
lgca.interaction_params['r_d'] = 0.01
lgca.interaction_params['r_b'] = 0.2
lgca.interaction_params['beta'] = 0.5
lgca.oxygen = np.zeros(lgca.cell_density.shape)
lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)

# grid mask
#lgca.mask = np.zeros_like(lgca.oxygen, dtype=bool)
#lgca.mask[1:-1:4] = True

# random mask
nx_full = lgca.cell_density.shape[0]
#print(nx_full)
lgca.mask = np.zeros((nx_full), dtype=bool)
p_vessel = 0.06
lgca.mask[1:-1] = lgca.rng.random((nx_full-2)) < p_vessel
#lgca.mask[-1]=True
lgca.original_mask = lgca.mask.copy()


lgca.oxygen[:] = convert_const*24 #60
#convert_const*24 # µmol/cm
#lgca.oxygen[-1]=convert_const*60
lgca.oxygen_source = convert_const*60 # µmol/cm

lgca.interaction_params['r_d_hypoxia'] = 0.5
lgca.interaction_params['max_ox'] = convert_const*95 # µmol/cm

lgca.interaction_params['lowox'] = convert_const*7
lgca.interaction_params['ultralowox'] = convert_const*0.7

lgca.interaction_params['kappa_clog'] = 0.35 #0.5 #0.3
lgca.interaction_params['theta_clog'] = 4 # 6 #12

# choose interaction
lgca.interaction = go_or_grow_kappa_chemo_mod
print("Interaction type used:", (lgca.interaction))

In [ ]:
import pandas as pd
import numpy as np

last_few = tmax-100
cell_density_txt = lgca.dens_t[last_few:tmax]
oxygen_txt = lgca.ox_t[last_few:tmax]
kappa_txt = np.array([lgca.calc_prop_mean(nodes=lgca.nodes_t[t],props=lgca.props,propname='kappa') for t in range(last_few, tmax)])
last_migrating_txt = lgca.channel_pop_t[last_few:tmax, :, :-1].sum(axis=-1) 
last_proliferating_txt = lgca.channel_pop_t[last_few:tmax, :, -1] 
nodes_txt = np.arange(cell_density_txt.shape[1])
time_txt = np.arange(last_few, tmax)

run_id = 128
T, X = np.meshgrid(time_txt, nodes_txt, indexing='ij')

df = pd.DataFrame({
    "time": T.ravel(),
    "node": X.ravel(),
    "density": cell_density_txt.ravel(),
    "oxygen": oxygen_txt.ravel(),
    "kappa": kappa_txt.ravel(),
    "migrating": last_migrating_txt.ravel(),
    "proliferating": last_proliferating_txt.ravel(),})

df = df.sort_values(by=["time", "node"], ascending=[False, True])

output_file = f"{output_dir}/lgca_spatiotemporal.xlsx"
sheet_name = f"run_{run_id}"

with pd.ExcelWriter(output_file, engine="openpyxl", mode="a" if os.path.exists(output_file) else "w") as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import types
import os    
output_dir = "04212026"
os.makedirs(output_dir, exist_ok=True)
#lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
#lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=True, recordfampop=False)
#O_history = lgca.ox_t


#O_final = lgca.ox_t[-1]   # last timestep, shape: (Nr_new,)
#print(lgca.oxygen)
#nodes_to_check = [1, 10, 50, 100, 150, 200, 250, 300, 400, 499]
    
#print(f"\nRun final oxygen values:")
#for n in nodes_to_check:
    #if n < len(O_final):
        #print(f"Node {n}: {O_final[n]}")
    #else:
        #print(f"Node {n}: out of range (max {len(O_final)-1})")


title_str = {run_id}
filename_tag = title_str
                
plt.figure()
plt.imshow(O_history, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='Oxygen')
plt.xlabel('Lattice node')
plt.ylabel('Time step')
plt.title(fr'Oxygen {title_str}')
plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
plt.close()
                
plt.figure()
lgca.plot_density(cmap='magma')
plt.title(fr'Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma',vmin=-4,vmax=4)
plt.title(fr'κ Distribution {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma',vmin=0,vmax=1)
plt.title(fr'θ Distribution {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_density(lgca.dead_dens_t,cmap='magma')
plt.title(fr'Dead Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
plt.close()
                
plt.figure()
lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma')
plt.title(fr'Proliferating Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma')
plt.title(fr'Migrating Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
plt.close()

# kappa histogram
mean_kappa = lgca.mean_prop_t['kappa']
all_vals_k = []
for t in range(tmax-100, tmax):
    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
    #print(f"t={t}, vals_k size={len(vals_k)}")
    all_vals_k.append(vals_k)
all_vals_k = np.concatenate(all_vals_k)
plt.figure()
plt.hist(all_vals_k, bins=30, linewidth=1)
plt.xlabel("κ")
plt.ylabel("Count")
plt.title((fr'Kappa Distribution {title_str}'))
plt.tight_layout()
plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
plt.close()
                
                
# theta histogram
mean_theta = lgca.mean_prop_t['theta']
all_vals_t = []
for t in range(tmax-100, tmax):
    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
    all_vals_t.append(vals_t)
all_vals_t = np.concatenate(all_vals_t)
plt.figure()
plt.hist(all_vals_t, bins=30, linewidth=1)
plt.xlabel("θ")
plt.ylabel("Count")
plt.title((fr'Theta Distribution {title_str}'))
plt.tight_layout()
plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
plt.close()

# heat map
# range on plot
#min_kappa = min(all_vals_k.min(), kappa_scan)
#max_kappa = max(all_vals_k.max(), kappa_scan)
#min_theta = min(all_vals_t.min(), theta_scan)
#max_theta = max(all_vals_t.max(), theta_scan)
plt.figure()
counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[-4, 4], [0, 1]], cmap='viridis')
plt.xlabel("κ")
plt.ylabel("θ")
#plt.xlim(min_kappa, max_kappa)
#plt.ylim(min_theta, max_theta)
plt.title(f"κ–θ Heat Map {title_str}")
plt.colorbar(im, label="Frequency")
#plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
plt.tight_layout()
plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
plt.close()

kappa_last = lgca.mean_prop_t['kappa'][-1, :]
nodes_kap = np.arange(kappa_last.shape[0])
plt.figure()
plt.plot(nodes, kappa_last, color='black')
plt.xlabel('Lattice node')
plt.ylabel(r'$\kappa$')
plt.title(fr'κ Spatial Profile {title_str}')
plt.xlim(0, l)
plt.tight_layout()
#plt.savefig(f"{output_dir}/kappa_spatial_end{filename_tag}.png", dpi=300)
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import os
import types
plt.close('all')
%matplotlib inline
output_dir = "01202026_figures_long"   
os.makedirs(output_dir, exist_ok=True)
    
kappa_min, kappa_max = -4.0, 4.0
theta_min, theta_max = 0.0, 1.0
    

kappa_clog_values = [0.5, 1.0, 1.5, 2.0]
theta_clog_values = [5, 6, 8, 10, 11, 12]
n_repeats = 5

run_id = 30

for kappaclog in kappa_clog_values:
    for thetaclog in theta_clog_values:
        for rep in range(n_repeats):

            run_id += 1
            print(run_id)
            
            # uniform dist of kappas and thetas
            kappa_scan = np.random.uniform(kappa_min, kappa_max, size=n_channels*k*capacity_param)
            theta_scan = np.random.uniform(theta_min, theta_max, size=n_channels*k*capacity_param)
            lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
            lgca.interaction_params['capacity'] = capacity_param
            lgca.interaction_params['kappa_std'] = 0.2
            lgca.interaction_params['r_d'] = 0.01
            lgca.interaction_params['r_b'] = 0.2
            lgca.interaction_params['beta'] = 0.5
        
            lgca.oxygen = np.zeros(lgca.cell_density.shape)
            lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)
                        
            nx_full= lgca.cell_density.shape[0]
            lgca.mask = np.zeros((nx_full), dtype=bool)
            p_vessel = 0.06
            lgca.mask[1:-1] = lgca.rng.random((nx_full-2)) < p_vessel
            lgca.original_mask = lgca.mask.copy()

            
            lgca.oxygen[:] = convert_const*24
            lgca.interaction_params['r_d_hypoxia'] = 0.5
            lgca.interaction_params['max_ox'] = convert_const*95
        
            lgca.interaction_params['lowox'] = convert_const*7
            lgca.interaction_params['ultralowox'] = convert_const*0.7
        
            lgca.interaction_params['kappa_clog'] = kappaclog
            lgca.interaction_params['theta_clog'] = thetaclog
            
            lgca.interaction = go_or_grow_kappa_chemo_mod
                        
                
            lgca.oxygen[:] = convert_const*24 # µmol/cm
            lgca.oxygen_source = convert_const*60 # µmol/cm
                        
        
        
            lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
            lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=False, recordfampop=False)
            O_history = lgca.ox_t
            title_str = fr'{run_id}'
            filename_tag = f"{run_id}_kc={kappaclog}_tc={thetaclog}"
                        
            
            plt.figure()
            lgca.plot_density(cmap='magma')
            plt.title(fr'Cell Density {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
            plt.close()
        
                    
            fig = plt.figure()
            lgca.plot_prop_spatial(propname='kappa', cmap='magma')
            plt.close(fig)
        
            fig2 = plt.figure()
            lgca.plot_prop_spatial(propname='theta', cmap='magma')
            plt.close(fig2)
                    
            migrating = lgca.channel_pop_t[..., :-1].sum(axis=(1, 2))
            proliferating = lgca.channel_pop_t[..., -1].sum(axis=1)
            total = migrating + proliferating
            pct_migrating = 100 * migrating / total
            pct_proliferating = 100 * proliferating / total
            tot = np.arange(lgca.channel_pop_t.shape[0])
            #plt.figure()
            #plt.plot(tot, pct_migrating, label='Migrating (%)')
            #plt.plot(tot, pct_proliferating, label='Proliferating (%)')
            #plt.xlabel('Time')
            #plt.ylabel('Percentage of cells')
            #plt.legend()
            #plt.tight_layout()
            #plt.savefig(f"{output_dir}/migration_vs_proliferation_{filename_tag}.png", dpi=300)
            #plt.close()
                        
            # kappa and theta plots
            mean_kappa = lgca.mean_prop_t['kappa']
            mean_theta = lgca.mean_prop_t['theta']
        
            # last 100 timesteps
            all_vals_k = []
            for t in range(tmax-100, tmax):
                vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                if len(vals_k) > 0:
                    all_vals_k.append(vals_k)
            all_vals_t = []
            for t in range(tmax-100, tmax):
                vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                if len(vals_t) > 0:
                    all_vals_t.append(vals_t)
            window_used = 100
                        
            # try last 200 if either is empty
            if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                all_vals_k = []
                for t in range(tmax-200, tmax):
                    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                    if len(vals_k) > 0:
                        all_vals_k.append(vals_k)
                all_vals_t = []
                for t in range(tmax-200, tmax):
                    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                    if len(vals_t) > 0:
                        all_vals_t.append(vals_t)
                window_used = 200
                        
            # still empty? skip
            if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                print(f"Skipping κ/θ histograms for {filename_tag}: no values in last 100 or 200 steps.")
                continue
                        
            all_vals_k = np.concatenate(all_vals_k)
            all_vals_t = np.concatenate(all_vals_t)
                        
            # calculate mean kappa, theta, and normalized oxygen for the last 100 (or 200) time steps
            # and plug into tanh switch function to calculate the switch
            mean_k_title = np.mean(all_vals_k)
            mean_t_title = np.mean(all_vals_t)
            O_history_last100 = lgca.ox_t[-100:, :]
            O_norm_mean = np.mean(O_history_last100 / lgca.interaction_params['max_ox'])
            tan_switch_title = 0.5 * (1 + np.tanh(mean_k_title * (O_norm_mean - mean_t_title)))
        
                        
            # kappa histogram
            #plt.figure()
            #plt.hist(all_vals_k, bins=30, linewidth=1, range=(-4,4))
            #plt.xlabel("κ")
            #plt.ylabel("Count")
            #plt.title(fr'Kappa Distribution {title_str} (last {window_used} steps), mean κ={mean_k_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
            #plt.tight_layout()
            #plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
            #plt.close()

            # theta histogram
            #plt.figure()
            #plt.hist(all_vals_t, bins=30, linewidth=1, range=(0, 1))
            #plt.xlabel("θ")
            #plt.ylabel("Count")
            #plt.title(fr'Theta Distribution {title_str} (last {window_used} steps), mean θ={mean_t_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
            #plt.tight_layout()
            #plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
            #plt.close()
                        
                        
            # heat map
            plt.figure()
            counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[-4,4], [0, 1]], cmap='viridis')
            plt.xlabel("κ")
            plt.ylabel("θ")
            plt.title(f"κ–θ Heat Map {title_str}")
            plt.colorbar(im, label="Frequency")
            #plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
            plt.close()

                                
print('finished :)')


In [ ]:
import matplotlib.pyplot as plt
import os
import types
import numpy as np

plt.close('all')
%matplotlib inline
output_dir = "01072026_figures"   
os.makedirs(output_dir, exist_ok=True)
kappa_vals=-2
theta_vals=0.8
filename_tag = f"_kappa={kappa_vals}_theta={theta_vals}"
O_local_vals = np.logspace(-9, -6, 100)
O_norm_vals = O_local_vals / lgca.interaction_params['max_ox']
tan_switch_vals = 0.5 * (1 + np.tanh(kappa_vals * (O_norm_vals - theta_vals)))
plt.scatter(O_norm_vals, tan_switch_vals, s=2)
plt.xscale("log")
plt.xlabel("Normalized Oxygen")
plt.ylabel("Tanh Switch")
plt.savefig(f"{output_dir}/tanh_function{filename_tag}.png", dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import os
import types
plt.close('all')
%matplotlib inline
output_dir = "12162025_random_kandt"   
os.makedirs(output_dir, exist_ok=True)
    
kappa_min, kappa_max = -4.0, 4.0
theta_min, theta_max = 0.0, 1.0
    
run_id = 0


    
for beta_scan in [0, 1]:
    for oxygen_scan in [convert_const*60, 0.5*convert_const*60]:
        for _ in range(16): 
            # uniform dist of kappas and thetas
            kappa_scan = np.random.uniform(kappa_min, kappa_max, size=3*8*12)
            theta_scan = np.random.uniform(theta_min, theta_max, size=3*8*12)
            lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
                
            lgca.interaction_params['capacity'] = capacity_param
            lgca.interaction_params['kappa_std'] = 0.2
            lgca.interaction_params['r_d'] = 0.01
            lgca.interaction_params['r_b'] = 0.2
            lgca.oxygen = np.zeros(lgca.cell_density.shape)
            lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)
                
            nx_full= lgca.cell_density.shape[0]
            #print(nx_full)
            lgca.mask = np.zeros((nx_full), dtype=bool)
            p_vessel = 0.06
            lgca.mask[1:-1] = lgca.rng.random((nx_full-2)) < p_vessel
            lgca.original_mask = lgca.mask.copy()
                
            lgca.oxygen[:] = convert_const*24
            lgca.interaction_params['r_d_hypoxia'] = 0.5
            lgca.interaction_params['max_ox'] = convert_const*95

            lgca.interaction_params['lowox'] = convert_const*7
            lgca.interaction_params['ultralowox'] = convert_const*0.7

            lgca.interaction_params['kappa_clog'] = 4
            lgca.interaction_params['theta_clog'] = 0.33 # clog around 1 cells?

            lgca.interaction_params['beta'] = beta_scan
                
            # choose interaction
            lgca.interaction = go_or_grow_kappa_chemo_mod
                
                
            lgca.oxygen[:] = convert_const*24 # µmol/cm
            lgca.oxygen_source = oxygen_scan # µmol/cm
                


            lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
            lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=False, recordfampop=False)
            O_history = lgca.ox_t
            title_str = fr'β={beta_scan}, θ={theta_scan}, κ={kappa_scan}, O2={oxygen_scan}'
            filename_tag = f"{run_id}_b{beta_scan}_o{oxygen_scan:.2e}".replace('.', 'p')
                
            plt.figure()
            plt.imshow(O_history, aspect='auto', origin='lower', cmap='magma')
            plt.colorbar(label='Oxygen')
            plt.xlabel('Lattice node')
            plt.ylabel('Time step')
            plt.title(fr'Oxygen {title_str}')
            plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
            plt.close()
                
            plt.figure()
            lgca.plot_density(cmap='magma')
            plt.title(fr'Cell Density {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
            plt.close()
        
            plt.figure()
            lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma')
            plt.title(fr'κ Distribution {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
            plt.close()
        
            plt.figure()
            lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma')
            plt.title(fr'θ Distribution {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
            plt.close()
        
            plt.figure()
            lgca.plot_density(lgca.dead_dens_t,cmap='magma')
            plt.title(fr'Dead Cell Density {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
            plt.close()
                
            plt.figure()
            lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma')
            plt.title(fr'Proliferating Cell Density {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
            plt.close()
        
            plt.figure()
            lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma')
            plt.title(fr'Migrating Cell Density {title_str}')
            plt.xlim(0, l)
            plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
            plt.close()

                
                
            # kappa and theta plots
            mean_kappa = lgca.mean_prop_t['kappa']
            mean_theta = lgca.mean_prop_t['theta']

            # last 100 timesteps
            all_vals_k = []
            for t in range(tmax-100, tmax):
                vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                if len(vals_k) > 0:
                    all_vals_k.append(vals_k)
            all_vals_t = []
            for t in range(tmax-100, tmax):
                vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                if len(vals_t) > 0:
                    all_vals_t.append(vals_t)
            window_used = 100
                
            # try last 200 if either is empty
            if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                all_vals_k = []
                for t in range(tmax-200, tmax):
                    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                    if len(vals_k) > 0:
                        all_vals_k.append(vals_k)
                all_vals_t = []
                for t in range(tmax-200, tmax):
                    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                    if len(vals_t) > 0:
                        all_vals_t.append(vals_t)
                window_used = 200
                
            # still empty? skip
            if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                print(f"Skipping κ/θ histograms for {filename_tag}: no values in last 100 or 200 steps.")
                continue
                
            all_vals_k = np.concatenate(all_vals_k)
            all_vals_t = np.concatenate(all_vals_t)
                
            # calculate mean kappa, theta, and normalized oxygen for the last 100 (or 200) time steps
            # and plug into tanh switch function to calculate the switch
            mean_k_title = np.mean(all_vals_k)
            mean_t_title = np.mean(all_vals_t)
            O_history_last100 = lgca.ox_t[-100:, :]
            O_norm_mean = np.mean(O_history_last100 / lgca.interaction_params['max_ox'])
            tan_switch_title = 0.5 * (1 + np.tanh(mean_k_title * (O_norm_mean - mean_t_title)))

                
            # kappa histogram
            plt.figure()
            plt.hist(all_vals_k, bins=30, linewidth=1, range=(-4,4))
            plt.xlabel("κ")
            plt.ylabel("Count")
            plt.title(fr'Kappa Distribution {title_str} (last {window_used} steps), mean κ={mean_k_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
            plt.tight_layout()
            plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
            plt.close()

            # theta histogram
            plt.figure()
            plt.hist(all_vals_t, bins=30, linewidth=1, range=(0, 1))
            plt.xlabel("θ")
            plt.ylabel("Count")
            plt.title(fr'Theta Distribution {title_str} (last {window_used} steps), mean θ={mean_t_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
            plt.tight_layout()
            plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
            plt.close()
                
                
            # heat map
            # range on plot
            #min_kappa = min(all_vals_k.min(), kappa_scan)
            #max_kappa = max(all_vals_k.max(), kappa_scan)
            #min_theta = min(all_vals_t.min(), theta_scan)
           # max_theta = max(all_vals_t.max(), theta_scan)
            plt.figure()
            counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[-4,4], [0, 1]], cmap='viridis')
            plt.xlabel("κ")
            plt.ylabel("θ")
            #plt.xlim(min_kappa, max_kappa)
            #plt.ylim(min_theta, max_theta)
            plt.title(f"κ–θ Heat Map {title_str}")
            plt.colorbar(im, label="Frequency")
            #plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
            plt.close()

            run_id +=1
                                
print('finished :)')


In [ ]:
import matplotlib.pyplot as plt
import os
import types
plt.close('all')
%matplotlib inline
output_dir = "12172025_figures"
os.makedirs(output_dir, exist_ok=True)

for kappa_scan in [0.5, 2, 4, 5]:
    for theta_scan in [0.2, 0.4, 0.6, 0.8]:
        for beta_scan in [0, 1]:
            for oxygen_scan in [convert_const*60,0.5*convert_const*60]:
                print(kappa_scan)
                
                lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
                # these give default values
                lgca.interaction_params['capacity'] = capacity_param
                lgca.interaction_params['kappa_std'] = 0.2
                lgca.interaction_params['r_d'] = 0.01
                lgca.interaction_params['r_b'] = 0.2
                lgca.oxygen = np.zeros(lgca.cell_density.shape)
                lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)
                
                # random mask
                nx_full= lgca.cell_density.shape[0]
                #print(nx_full)
                lgca.mask = np.zeros((nx_full), dtype=bool)
                p_vessel = 0.06
                lgca.mask[1:-1] = lgca.rng.random((nx_full-2)) < p_vessel
                lgca.original_mask = lgca.mask.copy()
                
                lgca.oxygen[:] = convert_const*24
                lgca.interaction_params['r_d_hypoxia'] = 0.5
                lgca.interaction_params['max_ox'] = convert_const*95

                lgca.interaction_params['lowox'] = convert_const*7
                lgca.interaction_params['ultralowox'] = convert_const*0.7

                lgca.interaction_params['kappa_clog'] = 10
                lgca.interaction_params['theta_clog'] = 0.08 # clog around 1 cells?

                lgca.interaction_params['beta'] = beta_scan
                
                # choose interaction
                lgca.interaction = go_or_grow_kappa_chemo_mod
                
                
                lgca.oxygen[:] = convert_const*24 # µmol/cm
                lgca.oxygen_source = oxygen_scan # µmol/cm
                


                lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
                lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=False, recordfampop=False)
                O_history = lgca.ox_t
                title_str = fr'β={beta_scan}, θ={theta_scan}, κ={kappa_scan}, O2={oxygen_scan}'
                filename_tag = f"b{beta_scan}_t{theta_scan}_k{kappa_scan}_o{oxygen_scan:.2e}".replace('.', 'p')
                
                plt.figure()
                plt.imshow(O_history, aspect='auto', origin='lower', cmap='magma')
                plt.colorbar(label='Oxygen')
                plt.xlabel('Lattice node')
                plt.ylabel('Time step')
                plt.title(fr'Oxygen {title_str}')
                plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
                plt.close()
                
                plt.figure()
                lgca.plot_density(cmap='magma')
                plt.title(fr'Cell Density {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma')
                plt.title(fr'κ Distribution {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma')
                plt.title(fr'θ Distribution {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_density(lgca.dead_dens_t,cmap='magma')
                plt.title(fr'Dead Cell Density {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
                plt.close()
                
                plt.figure()
                lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma')
                plt.title(fr'Proliferating Cell Density {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma')
                plt.title(fr'Migrating Cell Density {title_str}')
                plt.xlim(0, l)
                plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
                plt.close()

    
                # migrating: sum over space AND migrating channels
                migrating = lgca.channel_pop_t[..., :-1].sum(axis=(1, 2))
                
                # proliferating: sum over space
                proliferating = lgca.channel_pop_t[..., -1].sum(axis=1)
                
                total = migrating + proliferating
                
                pct_migrating = 100 * migrating / total
                pct_proliferating = 100 * proliferating / total
                
                t = np.arange(lgca.channel_pop_t.shape[0])
                

                plt.figure()
                plt.plot(tot, pct_migrating, label='Migrating (%)')
                plt.plot(t, pct_proliferating, label='Proliferating (%)')
                plt.xlabel('Time')
                plt.ylabel('Percentage of cells')
                plt.legend()
                plt.tight_layout()
                plt.savefig(f"{output_dir}/migration_vs_proliferation_{filename_tag}.png", dpi=300)
                plt.close()

                
                
                # kappa and theta plots
                mean_kappa = lgca.mean_prop_t['kappa']
                mean_theta = lgca.mean_prop_t['theta']

                # last 100 timesteps
                all_vals_k = []
                for t in range(tmax-100, tmax):
                    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                    if len(vals_k) > 0:
                        all_vals_k.append(vals_k)
                all_vals_t = []
                for t in range(tmax-100, tmax):
                    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                    if len(vals_t) > 0:
                        all_vals_t.append(vals_t)
                window_used = 100
                
                # try last 200 if either is empty
                if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                    all_vals_k = []
                    for t in range(tmax-200, tmax):
                        vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                        if len(vals_k) > 0:
                            all_vals_k.append(vals_k)
                    all_vals_t = []
                    for t in range(tmax-200, tmax):
                        vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                        if len(vals_t) > 0:
                            all_vals_t.append(vals_t)
                    window_used = 200
                
                # still empty? skip
                if len(all_vals_k) == 0 or len(all_vals_t) == 0:
                    print(f"Skipping κ/θ histograms for {filename_tag}: no values in last 100 or 200 steps.")
                    continue
                
                all_vals_k = np.concatenate(all_vals_k)
                all_vals_t = np.concatenate(all_vals_t)
                
                # calculate mean kappa, theta, and normalized oxygen for the last 100 (or 200) time steps
                # and plug into tanh switch function to calculate the switch
                mean_k_title = np.mean(all_vals_k)
                mean_t_title = np.mean(all_vals_t)
                O_history_last100 = lgca.ox_t[-100:, :]
                O_norm_mean = np.mean(O_history_last100 / lgca.interaction_params['max_ox'])
                tan_switch_title = 0.5 * (1 + np.tanh(mean_k_title * (O_norm_mean - mean_t_title)))

                
                # kappa histogram
                plt.figure()
                plt.hist(all_vals_k, bins=30, linewidth=1, range=(0, 5))
                plt.xlabel("κ")
                plt.ylabel("Count")
                plt.title(fr'Kappa Distribution {title_str} (last {window_used} steps), mean κ={mean_k_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
                plt.tight_layout()
                plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
                plt.close()

                # theta histogram
                plt.figure()
                plt.hist(all_vals_t, bins=30, linewidth=1, range=(0, 1))
                plt.xlabel("θ")
                plt.ylabel("Count")
                plt.title(fr'Theta Distribution {title_str} (last {window_used} steps), mean θ={mean_t_title:.2f}, tan_switch={tan_switch_title:.2f}', wrap=True)
                plt.tight_layout()
                plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
                plt.close()
                
                
                # heat map
                # range on plot
                min_kappa = min(all_vals_k.min(), kappa_scan)
                max_kappa = max(all_vals_k.max(), kappa_scan)
                min_theta = min(all_vals_t.min(), theta_scan)
                max_theta = max(all_vals_t.max(), theta_scan)
                plt.figure()
                counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[0, 5], [0, 1]], cmap='viridis')
                plt.xlabel("κ")
                plt.ylabel("θ")
                #plt.xlim(min_kappa, max_kappa)
                #plt.ylim(min_theta, max_theta)
                plt.title(f"κ–θ Heat Map {title_str}")
                plt.colorbar(im, label="Frequency")
                plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
                plt.tight_layout()
                plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
                plt.close()
                                
print('finished :)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import types
import os    
output_dir = "12162025_random_kandt"
os.makedirs(output_dir, exist_ok=True)
lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=True, recordfampop=False)
O_history = lgca.ox_t
title_str = 'test'
filename_tag = title_str
                
plt.figure()
plt.imshow(O_history, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='Oxygen')
plt.xlabel('Lattice node')
plt.ylabel('Time step')
plt.title(fr'Oxygen {title_str}')
plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
plt.close()
                
plt.figure()
lgca.plot_density(cmap='magma')
plt.title(fr'Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma')
plt.title(fr'κ Distribution {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma')
plt.title(fr'θ Distribution {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_density(lgca.dead_dens_t,cmap='magma')
plt.title(fr'Dead Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
plt.close()
                
plt.figure()
lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma')
plt.title(fr'Proliferating Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
plt.close()
        
plt.figure()
lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma')
plt.title(fr'Migrating Cell Density {title_str}')
plt.xlim(0, l)
plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
plt.close()

# kappa histogram
mean_kappa = lgca.mean_prop_t['kappa']
all_vals_k = []
for t in range(tmax-200, tmax):
    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
    #print(f"t={t}, vals_k size={len(vals_k)}")
    all_vals_k.append(vals_k)
all_vals_k = np.concatenate(all_vals_k)
plt.figure()
plt.hist(all_vals_k, bins=30, linewidth=1)
plt.xlabel("κ")
plt.ylabel("Count")
plt.title((fr'Kappa Distribution {title_str}'))
plt.tight_layout()
plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
plt.close()
                
                
# theta histogram
mean_theta = lgca.mean_prop_t['theta']
all_vals_t = []
for t in range(tmax-200, tmax):
    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
    all_vals_t.append(vals_t)
all_vals_t = np.concatenate(all_vals_t)
plt.figure()
plt.hist(all_vals_t, bins=30, linewidth=1)
plt.xlabel("θ")
plt.ylabel("Count")
plt.title((fr'Theta Distribution {title_str}'))
plt.tight_layout()
plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
plt.close()

# heat map
# range on plot
#min_kappa = min(all_vals_k.min(), kappa_scan)
#max_kappa = max(all_vals_k.max(), kappa_scan)
#min_theta = min(all_vals_t.min(), theta_scan)
#max_theta = max(all_vals_t.max(), theta_scan)
plt.figure()
counts, xedges, yedges, im = plt.hist2d(all_vals_k, all_vals_t, bins=80, range=[[-4, 4], [0, 1]], cmap='viridis')
plt.xlabel("κ")
plt.ylabel("θ")
#plt.xlim(min_kappa, max_kappa)
#plt.ylim(min_theta, max_theta)
plt.title(f"κ–θ Heat Map {title_str}")
plt.colorbar(im, label="Frequency")
plt.plot(kappa_scan, theta_scan, marker='*', markersize=10, color='red')
plt.tight_layout()
plt.savefig(f"{output_dir}/kappa_theta_heatmap_{filename_tag}.png", dpi=300)
plt.close()

In [ ]:
kappa_scan = 2
theta_scan = 0.5
lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
# these give default values
lgca.interaction_params['capacity'] = capacity_param
lgca.interaction_params['kappa_std'] = 0.2
lgca.interaction_params['r_d'] = 0.01
lgca.interaction_params['r_b'] = 0.2
lgca.interaction_params['beta'] = 0.5
lgca.oxygen = np.zeros(lgca.cell_density.shape)
lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)

lgca.mask = np.zeros_like(lgca.oxygen, dtype=bool)
lgca.mask[1:-1:4] = True


lgca.oxygen[:] = convert_const*24 # µmol/cm
lgca.oxygen_source = convert_const*60 # µmol/cm

lgca.interaction_params['r_d_hypoxia'] = 0.5
lgca.interaction_params['max_ox'] = convert_const*95 # µmol/cm

lgca.interaction_params['lowox'] = convert_const*7
lgca.interaction_params['ultralowox'] = convert_const*0.7

lgca.interaction_params['kappa_clog'] = 10
lgca.interaction_params['theta_clog'] = 0.08 # clog around 1 cells?

# choose interaction
lgca.interaction = go_or_grow_kappa_chemo_mod
print("Interaction type used:", (lgca.interaction))

In [ ]:
# kappa histogram
                mean_kappa = lgca.mean_prop_t['kappa']
                all_vals_k = []
                for t in range(tmax-100, tmax):
                    vals_k = mean_kappa[t].compressed() if hasattr(mean_kappa[t], "compressed") else mean_kappa[t]
                    #print(f"t={t}, vals_k size={len(vals_k)}")
                    all_vals_k.append(vals_k)
                all_vals_k = np.concatenate(all_vals_k)
                plt.figure()
                plt.hist(all_vals_k, bins=30, linewidth=1)
                plt.xlabel("κ")
                plt.ylabel("Count")
                plt.title((fr'Kappa Distribution {title_str}'))
                plt.tight_layout()
                plt.savefig(f"{output_dir}/kappa_hist_all_{filename_tag}.png", dpi=300)
                plt.close()
                                
                # theta histogram
                mean_theta = lgca.mean_prop_t['theta']
                all_vals_t = []
                for t in range(tmax-100, tmax):
                    vals_t = mean_theta[t].compressed() if hasattr(mean_theta[t], "compressed") else mean_theta[t]
                    all_vals_t.append(vals_t)
                all_vals_t = np.concatenate(all_vals_t)
                plt.figure()
                plt.hist(all_vals_t, bins=30, linewidth=1)
                plt.xlabel("θ")
                plt.ylabel("Count")
                plt.title((fr'Theta Distribution {title_str}'))
                plt.tight_layout()
                plt.savefig(f"{output_dir}/theta_hist_all_{filename_tag}.png", dpi=300)
                plt.close()

In [ ]:
plt.figure()
lgca.plot_density(cmap='magma')
plt.title('Cell Density')
plt.xlim(0, 100)

In [ ]:
plt.figure()
lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma')
plt.xlim(0, 100)

In [ ]:
plt.clf() 
plt.figure()
lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma')
plt.xlim(0, 200)

In [ ]:
# plot cell distribution
#plt.plot(lgca.cell_density)
#plt.xlabel('Lattice node')
#plt.ylabel('Cell density')
#plt.title('Cell distribution')
#plt.show()

In [ ]:
# dead
plt.figure()
lgca.plot_density(lgca.dead_dens_t,cmap='magma')
plt.title('Dead Cell Density')
plt.xlim(0, 100)

In [ ]:
# proliferating
lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma', figsize=(10,10))

In [ ]:
# migrating
lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma', figsize=(10,10))

In [ ]:
import matplotlib.pyplot as plt
import os
import types
plt.close('all')
%matplotlib inline
output_dir = "11date2025_figures"
os.makedirs(output_dir, exist_ok=True)

for kappa_scan in [2]: #[0.5, 2, 4, 5]:
    for theta_scan in [0.5]: #[0.2, 0.4, 0.6, 0.8]:
        for beta_scan in [1]:#[0, 1]:
            for oxygen_scan in [1.974e-6]:#[1.974e-6, 0.5*1.974e-6]:
                print(kappa_scan)
                
                lgca = get_lgca(ib=True, ve = False, nodes=nodes, geometry='1d', bc='reflect', interaction='go_or_grow', kappa=kappa_scan, theta=theta_scan)
                # these give default values
                lgca.interaction_params['capacity'] = 100
                lgca.interaction_params['kappa_std'] = 0.2
                lgca.interaction_params['r_d'] = 0.01
                lgca.interaction_params['r_b'] = 0.2
                lgca.oxygen = np.zeros(lgca.cell_density.shape)
                lgca.dead_cell_density = np.zeros(lgca.cell_density.shape)
                lgca.oxygen[:] = 7.895e-7 # micro-mol/cm
                lgca.interaction_params['r_d_hypoxia'] = 0.5
                lgca.interaction_params['max_ox'] = 3.126e-6 # micro-mol/cm
                # choose interaction
                lgca.interaction = go_or_grow_kappa_chemo_mod

                lgca.interaction_params['beta'] = beta_scan
                lgca.oxygen_source = oxygen_scan #oxygen source (vessel) every 200 um
                
                #print('kappa=', kappa_scan)
                #print('theta=', theta_scan)
                #print('beta=', beta_scan)
                #print('oxygen=', oxygen_scan)

                lgca.timeevo_mod = types.MethodType(timeevo_mod, lgca)
                lgca.timeevo_mod(timesteps=tmax, record=True, recordN=True, recorddens=True, recorddead=True, recordox=True, recordchanneldens=True, showprogress=False, recordfampop=False)
                O_history = lgca.ox_t
                title_str = fr'β={beta_scan}, θ={theta_scan}, κ={kappa_scan}, O2={oxygen_scan}'
                filename_tag = f"b{beta_scan}_t{theta_scan}_k{kappa_scan}_o{oxygen_scan:.2e}".replace('.', 'p')
                
                plt.figure()
                plt.imshow(O_history, aspect='auto', origin='lower', cmap='magma')
                plt.colorbar(label='Oxygen')
                plt.xlabel('Lattice node')
                plt.ylabel('Time step')
                plt.title(fr'Oxygen {title_str}')
                plt.savefig(f"{output_dir}/oxygen_{filename_tag}.png", dpi=300)
                plt.close()
                
                plt.figure()
                lgca.plot_density(cmap='magma')
                plt.title(fr'Cell Density {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/density_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_prop_spatial(propname='kappa', cbarlabel=r'Switch parameter $\kappa$', cmap='magma')
                plt.title(fr'κ Distribution {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/kappa_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_prop_spatial(propname='theta', cbarlabel=r'Switch parameter $\theta$', cmap='magma')
                plt.title(fr'θ Distribution {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/theta_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_density(lgca.dead_dens_t,cmap='magma')
                plt.title(fr'Dead Cell Density {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/dead_{filename_tag}.png", dpi=300)
                plt.close()
                
                plt.figure()
                lgca.plot_density(lgca.channel_pop_t[...,-1],cmap='magma')
                plt.title(fr'Proliferating Cell Density {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/proliferating_{filename_tag}.png", dpi=300)
                plt.close()
        
                plt.figure()
                lgca.plot_density(lgca.channel_pop_t[...,:-1].sum(axis=-1),cmap='magma')
                plt.title(fr'Migrating Cell Density {title_str}')
                plt.xlim(0, 300)
                plt.savefig(f"{output_dir}/migrating_{filename_tag}.png", dpi=300)
                plt.close()
print('finished :)')